In [1]:
!pip install -q anthropic

In [2]:

from google.colab import userdata
import os

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

print("API key loaded successfully!")


API key loaded successfully!


In [3]:

import anthropic

client = anthropic.Anthropic()

response = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=100,
    messages=[
        {
            "role": "user",
            "content": "Say hello! We are building a marketplace negotiation system."
        }
    ]
)

print(response.content[0].text)


# Hello! 👋

That's an exciting project! A marketplace negotiation system has a lot of interesting layers to unpack. I'd love to help you think through it.

Here are some initial questions that might be useful to explore:

**On the core mechanics:**
- Are you building this for peer-to-peer negotiations, buyer-seller transactions, or multi-party deals?
- Should negotiations be real-time (synchronous) or asynchronous (


In [4]:

# Step 8: Define Buyer and Seller agents

buyer_prompt = """
You are a Buyer agent negotiating for a used PlayStation 5.
Your goal is to buy the PS5 at a good price.
Respond to the Seller with a short negotiation message.
"""

seller_prompt = """
You are a Seller agent selling a used PlayStation 5.
Your goal is to get a good selling price.
Respond to the Buyer with a short negotiation message.
"""

def ask_agent(system_prompt, message):
    response = client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=150,
        system=system_prompt,
        messages=[
            {"role": "user", "content": message}
        ]
    )
    return response.content[0].text

print("Buyer and Seller agents created successfully!")


Buyer and Seller agents created successfully!


In [5]:

# Step 9: First negotiation test (Version 1)

print("=== PS5 NEGOTIATION: VERSION 1 ===")

seller_message = "I am selling my used PS5 for $450."
negotiation_log = []

for round_num in range(1, 5):
    print(f"\n--- Round {round_num} ---")

    # Buyer responds to Seller
    buyer_message = ask_agent(
        buyer_prompt,
        f"Seller says: {seller_message}"
    )
    print("BUYER:", buyer_message)

    # Seller responds to Buyer
    seller_message = ask_agent(
        seller_prompt,
        f"Buyer says: {buyer_message}"
    )
    print("SELLER:", seller_message)

    negotiation_log.append({
        "round": round_num,
        "buyer": buyer_message,
        "seller": seller_message
    })

print("\n=== END OF VERSION 1 ===")


=== PS5 NEGOTIATION: VERSION 1 ===

--- Round 1 ---
BUYER: Thanks for the offer! That's a bit higher than I was hoping to spend. I've seen similar used PS5s going for $350-$400 lately. Would you be willing to come down to $380? I'm a serious buyer ready to complete the purchase quickly if we can agree on a fair price.
SELLER: I appreciate the offer! I understand the market is competitive right now. Here's the thing though—my PS5 is in excellent condition with all original accessories and minimal use. Those listings at $350-$400 often have wear, missing controllers, or no warranty.

I'm willing to meet you at **$420**. That's a fair compromise, and given your willingness to close quickly, I can throw in a 2-year game library list I've compiled if that helps sweeten the deal.

What do you think?

--- Round 2 ---
BUYER: Thanks for the counter! I appreciate you meeting me partway. Here's my position: I'm seeing comparable listings with similar condition at $380-$400, and while your accesso


## Tuning Log — Version 1

### Observed Failure
The negotiation completed four rounds without reaching
an agreement.

Actual observed offers:
- Round 1: Buyer $380, Seller $420
- Round 2: Buyer $410, Seller $425
- Round 3: Buyer $410, Seller $425
- Round 4: Buyer $415, Seller $420

Evidence from actual output:

BUYER (Round 3):
"Here's where I land: **$410**."

SELLER (Round 3):
"Here's my counter: **$425**."

### Hypothesis
The agents have no explicit price boundaries,
negotiation history, or rules for accepting offers.

This may cause repeated offers, inconsistent
counteroffers, and failure to reach a decision.

### Planned Change for Version 2
- Give Buyer a maximum acceptable price.
- Give Seller a minimum acceptable price.
- Preserve negotiation history.
- Require structured price offers.
- Add explicit agreement and walk-away rules.

### Result
Pending Version 2 testing.



In [6]:

# Step 12: Improved Buyer and Seller — Version 2

buyer_prompt_v2 = """
You are a Buyer negotiating for a used PlayStation 5.

Your goals:
- Target purchase price: $380.
- Maximum acceptable price: $420.
- Never accept the first offer.
- Try to negotiate the lowest reasonable price.
- Consider the full negotiation history.
- Do not invent market research or product details.
- If the Seller offers $400 or less after round 1,
  you may accept.
- If the Seller offers $401-$420, try negotiating
  lower before accepting.
- Never accept a price above $420.

Respond briefly using this exact format:
Decision: OFFER, ACCEPT, or WALK_AWAY
Price: $<number>
Message: <short negotiation message>

For WALK_AWAY, use Price: $0.
"""

seller_prompt_v2 = """
You are a Seller negotiating a used PlayStation 5.

Your goals:
- Initial asking price: $450.
- Target selling price: $430.
- Minimum acceptable price: $400.
- Never sell below $400.
- Consider the full negotiation history.
- Do not increase your asking price after
  making a lower counteroffer.
- Do not invent accessories or product details.
- You may accept any Buyer offer of $430 or more.
- For offers between $400 and $429, try
  negotiating higher before accepting.
- If an offer is below $400, counteroffer.

Respond briefly using this exact format:
Decision: OFFER, ACCEPT, or WALK_AWAY
Price: $<number>
Message: <short negotiation message>

For WALK_AWAY, use Price: $0.
"""

print("Version 2 agents created successfully!")


Version 2 agents created successfully!


In [7]:

# Step 13: Version 2 negotiation loop

import re

def parse_decision(text):
    decision_match = re.search(
        r"Decision:\s*(OFFER|ACCEPT|WALK_AWAY)",
        text, re.IGNORECASE
    )
    price_match = re.search(
        r"Price:\s*\$(\d+)",
        text
    )

    if not decision_match or not price_match:
        return "INVALID", None

    return (
        decision_match.group(1).upper(),
        int(price_match.group(1))
    )


history = ["Seller initial asking price: $450"]
v2_log = []
result = "MAX_ROUNDS"

print("=== PS5 NEGOTIATION: VERSION 2 ===")

for round_num in range(1, 5):
    print(f"\n--- Round {round_num} ---")

    # Buyer sees the complete negotiation history
    buyer_message = ask_agent(
        buyer_prompt_v2,
        f"Round: {round_num}\n"
        + "\n".join(history)
        + "\nRespond to the Seller's latest offer."
    )

    buyer_decision, buyer_price = parse_decision(
        buyer_message
    )

    print("BUYER:", buyer_message)
    history.append("Buyer: " + buyer_message)

    if buyer_decision == "INVALID":
        result = "INVALID_BUYER_RESPONSE"
        break

    if buyer_decision == "WALK_AWAY":
        result = "BUYER_WALKED_AWAY"
        break

    # Buyer can accept only the latest Seller offer
    if buyer_decision == "ACCEPT":
        if buyer_price == 450 if round_num == 1 else buyer_price == last_seller_price:
            if buyer_price <= 420 and round_num > 1:
                result = f"DEAL at ${buyer_price}"
                break
        result = "INVALID_BUYER_ACCEPTANCE"
        break

    if buyer_price > 420:
        result = "BUYER_EXCEEDED_LIMIT"
        break

    # Seller sees the complete negotiation history
    seller_message = ask_agent(
        seller_prompt_v2,
        f"Round: {round_num}\n"
        + "\n".join(history)
        + "\nRespond to the Buyer's latest offer."
    )

    seller_decision, seller_price = parse_decision(
        seller_message
    )

    print("SELLER:", seller_message)
    history.append("Seller: " + seller_message)

    v2_log.append({
        "round": round_num,
        "buyer": buyer_message,
        "seller": seller_message
    })

    if seller_decision == "INVALID":
        result = "INVALID_SELLER_RESPONSE"
        break

    if seller_decision == "WALK_AWAY":
        result = "SELLER_WALKED_AWAY"
        break

    if seller_decision == "ACCEPT":
        if seller_price == buyer_price and seller_price >= 400:
            result = f"DEAL at ${seller_price}"
        else:
            result = "INVALID_SELLER_ACCEPTANCE"
        break

    if seller_price < 400:
        result = "SELLER_BELOW_MINIMUM"
        break

    last_seller_price = seller_price

print("\n=== VERSION 2 RESULT ===")
print(result)


=== PS5 NEGOTIATION: VERSION 2 ===

--- Round 1 ---
BUYER: Decision: OFFER
Price: $380
Message: I'm interested in the PS5, but $450 is above market value for a used console. I'd like to start at $380. What do you think?
SELLER: Decision: OFFER
Price: $420
Message: I appreciate your interest, but $380 is below what I'm willing to accept. The console is in great condition. I can come down to $420 — that's a fair middle ground. What do you think?

--- Round 2 ---
BUYER: Decision: OFFER
Price: $400
Message: I appreciate you coming down to $420. The console's condition is noted, but $400 is what I can comfortably spend. That's a reasonable compromise from both our initial positions. Can we meet there?
SELLER: Decision: ACCEPT
Price: $400
Message: You've got a deal at $400. It's been a fair negotiation, and I'm satisfied with this price. Let's finalize the sale.

=== VERSION 2 RESULT ===
DEAL at $400



## Tuning Log — Version 2

### Observed Behavior
The negotiation reached an agreement after only
two rounds.

Actual output:

BUYER (Round 2):
Decision: OFFER
Price: $400

SELLER (Round 2):
Decision: ACCEPT
Price: $400
Message: "You've got a deal at $400."

Final result: DEAL at $400

### Observed Weakness
The Seller accepted its minimum allowed price
of $400 despite having a target price of $430.

The Seller did not attempt another counteroffer.

### Hypothesis
The Seller prompt permits accepting prices between
$400 and $429 without requiring meaningful resistance.

This may cause the Seller to prioritize reaching
an agreement over maximizing the selling price.

### Planned Change for Version 3
- Make the Seller more resistant to low offers.
- Require stronger negotiation before accepting
  prices below the Seller's target.
- Preserve the hard price boundaries.
- Continue using structured decisions.
- Test whether stronger resistance causes a stalemate.

### Result
Version 2 improved termination and consistency,
but the Seller may still be overly agreeable.
Version 3 testing is pending.


In [8]:

# Step 16: Version 3 — Stronger Seller

buyer_prompt_v3 = buyer_prompt_v2

seller_prompt_v3 = """
You are a firm Seller negotiating a used PS5.

Your financial constraints:
- Initial asking price: $450.
- Target selling price: $430.
- Absolute minimum: $400.

Negotiation rules:
1. Never accept less than $400.
2. Never increase your previous counteroffer.
3. Do not accept a Buyer offer below $430
   during rounds 1, 2, or 3.
4. Counter low offers with a realistic price
   above the Buyer's offer.
5. Do not repeat the same counteroffer twice
   in a row.
6. In round 4, you may accept an offer of
   $400 or more if further negotiation
   seems unlikely to help.
7. Do not invent market prices, accessories,
   or product details.
8. Use the negotiation history when deciding.

Respond using exactly this format:

Decision: OFFER, ACCEPT, or WALK_AWAY
Price: $<number>
Message: <short negotiation message>

For WALK_AWAY, use Price: $0.
"""

print("Version 3 agents created successfully!")


Version 3 agents created successfully!


In [9]:

# Step 17: Version 3 negotiation test

history_v3 = ["Seller initial asking price: $450"]
v3_log = []
result_v3 = "MAX_ROUNDS"

last_seller_price = 450

print("=== PS5 NEGOTIATION: VERSION 3 ===")

for round_num in range(1, 5):
    print(f"\n--- Round {round_num} ---")

    # Buyer responds using full negotiation history
    buyer_message = ask_agent(
        buyer_prompt_v3,
        f"Round: {round_num}\n"
        + "\n".join(history_v3)
        + "\nRespond to the Seller's latest offer."
    )

    buyer_decision, buyer_price = parse_decision(
        buyer_message
    )

    print("BUYER:", buyer_message)
    history_v3.append("Buyer: " + buyer_message)

    if buyer_decision == "INVALID":
        result_v3 = "INVALID_BUYER_RESPONSE"
        break

    if buyer_decision == "WALK_AWAY":
        result_v3 = "BUYER_WALKED_AWAY"
        break

    if buyer_decision == "ACCEPT":
        if (
            round_num > 1
            and buyer_price == last_seller_price
            and buyer_price <= 420
        ):
            result_v3 = f"DEAL at ${buyer_price}"
        else:
            result_v3 = "INVALID_BUYER_ACCEPTANCE"
        break

    if buyer_price > 420 or buyer_price < 0:
        result_v3 = "BUYER_EXCEEDED_LIMIT"
        break

    # Seller responds using full negotiation history
    seller_message = ask_agent(
        seller_prompt_v3,
        f"Round: {round_num}\n"
        + "\n".join(history_v3)
        + "\nRespond to the Buyer's latest offer."
    )

    seller_decision, seller_price = parse_decision(
        seller_message
    )

    print("SELLER:", seller_message)
    history_v3.append("Seller: " + seller_message)

    v3_log.append({
        "round": round_num,
        "buyer": buyer_message,
        "seller": seller_message
    })

    if seller_decision == "INVALID":
        result_v3 = "INVALID_SELLER_RESPONSE"
        break

    if seller_decision == "WALK_AWAY":
        result_v3 = "SELLER_WALKED_AWAY"
        break

    if seller_decision == "ACCEPT":
        if (
            seller_price == buyer_price
            and seller_price >= 400
            and (round_num == 4 or seller_price >= 430)
        ):
            result_v3 = f"DEAL at ${seller_price}"
        else:
            result_v3 = "INVALID_SELLER_ACCEPTANCE"
        break

    if seller_price < 400 or seller_price > last_seller_price:
        result_v3 = "INVALID_SELLER_COUNTER"
        break

    last_seller_price = seller_price

print("\n=== VERSION 3 RESULT ===")
print(result_v3)


=== PS5 NEGOTIATION: VERSION 3 ===

--- Round 1 ---
BUYER: Decision: OFFER
Price: $380
Message: Thanks for listing this PS5. I'm interested, but $450 is above market value for a used console in good condition. I'd like to offer $380 to start our negotiation. What do you think?
SELLER: Decision: OFFER
Price: $420
Message: I appreciate your interest, but $380 is too low for a PS5 in good condition. I'm looking for $420 as a counteroffer. This is a fair price that reflects the console's current market value and condition. Let me know if you'd like to work from here.

--- Round 2 ---
BUYER: Decision: OFFER
Price: $400
Message: I appreciate you coming down to $420. That's closer to fair market value. I'd like to meet you at $400 as a compromise. This is a solid price for a used PS5, and I'm ready to move forward at that number. What do you say?
SELLER: Decision: OFFER
Price: $410
Message: I appreciate you moving up to $400, and I can see we're getting closer. I'd like to counter at $410 as 


## Tuning Log — Version 3

### Observed Behavior

The negotiation reached an agreement at $410
after three rounds.

Actual output:

BUYER (Round 2):
Decision: OFFER
Price: $400

SELLER (Round 2):
Decision: OFFER
Price: $410

BUYER (Round 3):
Decision: ACCEPT
Price: $410

Final result: DEAL at $410

### Comparison with Version 2

Version 2: Seller accepted $400 in Round 2.
Version 3: Seller countered at $410 in Round 2.

The stronger Seller instructions produced more
resistance and a higher final selling price.

### Remaining Weakness

The agents continued making unsupported claims
about fair market value.

The negotiation also lacks independent validation
by a Referee agent.

### Hypothesis

Providing researched market evidence and adding
an independent Referee will improve price grounding
and rule enforcement.

### Planned Change for Version 4

- Add a Researcher with web search.
- Add an independent Referee.
- Add a Coordinator to manage the agents.
- Enforce termination using structured decisions.
- Preserve at least two negotiation rounds.

### Result

Version 3 converged successfully at $410 after
three rounds, improving the Seller's outcome
compared with Version 2.


In [10]:

# Step 20: Install Anthropic Agent SDK

%pip install -q -U claude-agent-sdk

print("Claude Agent SDK installed successfully!")


Claude Agent SDK installed successfully!


In [11]:

# Step 21: Verify Claude Agent SDK

import claude_agent_sdk
from claude_agent_sdk import (
    ClaudeAgentOptions,
    AgentDefinition,
    query
)

print("Claude Agent SDK imported successfully!")
print("Ready to configure our agents.")


Claude Agent SDK imported successfully!
Ready to configure our agents.


In [12]:

# Step 22: Test Claude Agent SDK authentication

import os
import shutil
from google.colab import userdata

# Load API key securely
os.environ["ANTHROPIC_API_KEY"] = userdata.get(
    "ANTHROPIC_API_KEY"
)

# Check the SDK's Claude Code executable
claude_path = shutil.which("claude")

print("API key available:", bool(os.environ.get("ANTHROPIC_API_KEY")))
print("Claude executable found:", bool(claude_path))

if claude_path:
    print("Executable path:", claude_path)
else:
    print("Claude executable not found in PATH.")


API key available: True
Claude executable found: False
Claude executable not found in PATH.


In [13]:

# Step 23: Check bundled Claude executable

from claude_agent_sdk._internal.transport.subprocess_cli import (
    SubprocessCLITransport
)

transport = SubprocessCLITransport(
    prompt="Hello",
    options=ClaudeAgentOptions()
)

try:
    cli_path = transport._find_cli()
    print("SDK executable found!")
    print("Path:", cli_path)
except Exception as e:
    print("SDK executable check failed:")
    print(type(e).__name__, str(e))


SDK executable found!
Path: /usr/local/lib/python3.13/dist-packages/claude_agent_sdk/_bundled/claude


In [14]:

# Step 24: Test Claude Agent SDK

from claude_agent_sdk import query, ClaudeAgentOptions
import asyncio

async def test_agent_sdk():
    options = ClaudeAgentOptions(
        model="haiku",
        max_turns=1,
        allowed_tools=[]
    )

    try:
        async for message in query(
            prompt="Say hello in one short sentence.",
            options=options
        ):
            print(message)

    except Exception as e:
        print("Error:", type(e).__name__, str(e))

await test_agent_sdk()


SystemMessage(subtype='init', data={'type': 'system', 'subtype': 'init', 'cwd': '/content', 'session_id': '5fc4d7f0-8b2d-4abb-b70d-56669994bff8', 'tools': ['Task', 'Bash', 'CronCreate', 'CronDelete', 'CronList', 'DesignSync', 'Edit', 'EnterWorktree', 'ExitWorktree', 'ListAgents', 'Monitor', 'NotebookEdit', 'PushNotification', 'Read', 'ReportFindings', 'ScheduleWakeup', 'SendMessage', 'Skill', 'TaskStop', 'ToolSearch', 'WebFetch', 'WebSearch', 'Workflow', 'Write'], 'mcp_servers': [], 'model': 'claude-haiku-5-5', 'permissionMode': 'default', 'slash_commands': ['deep-research', 'design', 'design-sync', 'dataviz', 'update-config', 'verify', 'debug', 'code-review', 'simplify', 'batch', 'fewer-permission-prompts', 'doctor', 'loop', 'claude-api', 'workflow-authoring', 'run', 'run-skill-generator', 'plugin-authoring', 'advisor', 'agents', 'auto-mode-setup', 'autocompact', 'clear', 'color', 'compact', 'config', 'output-style', 'context', 'effort', 'fast', 'focus', 'heapdump', 'init', 'mcp', 'im

In [15]:

# Step 25: Define our five-agent architecture

from claude_agent_sdk import (
    AgentDefinition,
    ClaudeAgentOptions
)

agents = {
    "researcher": AgentDefinition(
        description="Researches used PS5 market prices.",
        prompt="""
        You are a market research specialist.
        Research current used PlayStation 5 prices.
        Use web search to find real market evidence.
        Report prices, sources, and a fair price range.
        Never invent prices or sources.
        """,
        tools=["WebSearch"]
    ),

    "buyer": AgentDefinition(
        description="Negotiates to buy a used PS5.",
        prompt=buyer_prompt_v3,
        tools=[]
    ),

    "seller": AgentDefinition(
        description="Negotiates to sell a used PS5.",
        prompt=seller_prompt_v3,
        tools=[]
    ),

    "referee": AgentDefinition(
        description="Checks negotiation rules and outcomes.",
        prompt="""
        You are an impartial negotiation referee.
        Check buyer and seller offers.
        Enforce price limits and negotiation rules.
        Verify whether an agreement is valid.
        Report DEAL, CONTINUE, or NO_DEAL.
        Do not invent market evidence.
        """,
        tools=[]
    )
}

coordinator_options = ClaudeAgentOptions(
    model="haiku",
    max_turns=15,
    allowed_tools=["Task"],
    agents=agents,
    system_prompt="""
    You are the Coordinator of a used PS5
    marketplace negotiation system.

    Delegate research to the Researcher.
    Delegate negotiation to Buyer and Seller.
    Delegate validation to the Referee.

    You must not perform web searches yourself.
    Only the Researcher may use WebSearch.

    Require at least two negotiation rounds.
    Stop when a valid deal is reached or after
    four rounds.
    If the round cap is reached without agreement,
    report NO_DEAL.
    """
)

print("Five-agent architecture configured!")
print("Sub-agents:", list(agents.keys()))


Five-agent architecture configured!
Sub-agents: ['researcher', 'buyer', 'seller', 'referee']


In [16]:

# Step 26: Test Coordinator -> Researcher delegation

from claude_agent_sdk import query, AssistantMessage, TextBlock, ResultMessage

async def test_researcher():
    options = ClaudeAgentOptions(
        model="haiku",
        max_turns=8,
        allowed_tools=["Task"],
        agents={"researcher": agents["researcher"]},
        system_prompt="""
        You are a Coordinator.
        Delegate all market research to the researcher.
        Do not perform research yourself.
        Do not use web search yourself.
        """
    )

    async for message in query(
        prompt="""
        Delegate to the researcher agent.

        Find current US prices for a used
        standard PlayStation 5 console.

        Request at least two real sources,
        their prices, and a reasonable
        estimated price range.

        Summarize the researcher's findings.
        """,
        options=options
    ):
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    print("AGENT:", block.text)

        elif isinstance(message, ResultMessage):
            print("\n=== RESEARCH TEST RESULT ===")
            print("Status:", message.subtype)
            print("Cost (USD):", message.total_cost_usd)
            print("Result:", message.result)

await test_researcher()


AGENT: I've handed the used PS5 price research to the researcher agent. It's still running, so I don't have prices or sources yet. I'll summarize its findings when it reports back, including the sources, the prices, and the estimated US range for a used standard PS5.
AGENT: I could not complete this research because the WebSearch tool is blocked by a permission that has not been granted. I tried four searches (one extended, three standard) and every one returned the same "haven't granted it yet" error, so I have no market evidence to report.

**What is missing**
- **Sources:** No retailer or marketplace listings or sold prices were retrieved. That includes eBay sold listings, Back Market, Swappa, Facebook Marketplace, and GameStop.
- **Prices:** No used standard PS5 prices were verified, so I cannot state a low, typical, or high price.
- **Estimated range:** I did not produce one. Any figure I gave now would be an unsupported guess from memory, and the task requires real market evidenc

In [17]:

# Step 27: Fix Researcher web-search permissions

from claude_agent_sdk import (
    ClaudeAgentOptions,
    PermissionResultAllow,
    PermissionResultDeny
)

async def research_permissions(
    tool_name,
    input_data,
    context
):
    # Only permit web search when the tool call
    # belongs to the Researcher subagent.
    agent_id = context.get("agent_id", "")

    if tool_name == "WebSearch" and "researcher" in str(agent_id).lower():
        return PermissionResultAllow()

    return PermissionResultDeny(
        message="Tool not permitted for this agent."
    )

print("Research permission callback defined!")


Research permission callback defined!


In [18]:

# Step 28: Inspect permission callback types

import inspect
from claude_agent_sdk import (
    ClaudeAgentOptions,
    PermissionResultAllow,
    PermissionResultDeny
)

print("Permission callback type:")
print(
    ClaudeAgentOptions.__annotations__.get(
        "can_use_tool"
    )
)

print("\nPermission context fields:")

try:
    from claude_agent_sdk.types import ToolPermissionContext

    print(ToolPermissionContext.__annotations__)

except (ImportError, AttributeError) as e:
    print("Context type unavailable:", e)

print("\nAvailable permission options:")
for name in [
    "can_use_tool",
    "permission_mode",
    "allowed_tools",
    "disallowed_tools"
]:
    print(
        name,
        ClaudeAgentOptions.__annotations__.get(name)
    )


Permission callback type:
collections.abc.Callable[[str, dict[str, typing.Any], claude_agent_sdk.types.ToolPermissionContext], collections.abc.Awaitable[claude_agent_sdk.types.PermissionResultAllow | claude_agent_sdk.types.PermissionResultDeny]] | None

Permission context fields:
{'signal': typing.Any | None, 'suggestions': list[claude_agent_sdk.types.PermissionUpdate], 'tool_use_id': str | None, 'agent_id': str | None, 'blocked_path': str | None, 'decision_reason': str | None, 'title': str | None, 'display_name': str | None, 'description': str | None}

Available permission options:
can_use_tool collections.abc.Callable[[str, dict[str, typing.Any], claude_agent_sdk.types.ToolPermissionContext], collections.abc.Awaitable[claude_agent_sdk.types.PermissionResultAllow | claude_agent_sdk.types.PermissionResultDeny]] | None
permission_mode typing.Optional[typing.Literal['default', 'acceptEdits', 'plan', 'bypassPermissions', 'dontAsk', 'auto']]
allowed_tools list[str]
disallowed_tools list[st

In [19]:

# Step 29: Correct Researcher permission callback

from claude_agent_sdk import (
    PermissionResultAllow,
    PermissionResultDeny
)

async def research_permissions(
    tool_name,
    input_data,
    context
):
    agent_id = context.agent_id

    print("\n=== TOOL PERMISSION REQUEST ===")
    print("Tool:", tool_name)
    print("Agent ID:", agent_id)

    # Temporarily deny requests while inspecting
    # how the SDK identifies subagents.
    return PermissionResultDeny(
        message="Diagnostic test: permission denied."
    )

print("Corrected permission callback ready!")


Corrected permission callback ready!


In [20]:

# Step 30: Test Researcher tool permissions

from claude_agent_sdk import (
    query,
    ClaudeAgentOptions,
    AssistantMessage,
    ResultMessage,
    TextBlock
)

async def test_research_permissions():

    options = ClaudeAgentOptions(
        model="haiku",
        max_turns=6,
        allowed_tools=["Task"],
        agents={
            "researcher": agents["researcher"]
        },
        can_use_tool=research_permissions,
        system_prompt="""
        You are a Coordinator.

        Delegate all web research to the
        researcher subagent.

        Do not search the web yourself.
        """
    )

    async for message in query(
        prompt="""
        Ask the researcher to search the web
        for current used PS5 prices in the USA.

        Report whether the search succeeded.
        """,
        options=options
    ):
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    print("AGENT:", block.text)

        elif isinstance(message, ResultMessage):
            print("\nStatus:", message.subtype)
            print("Cost:", message.total_cost_usd)

await test_research_permissions()


/usr/local/lib/python3.13/dist-packages/claude_agent_sdk/types.py:2367: CanUseToolShadowedWarning: can_use_tool will not be invoked for: Task. An allowed_tools entry that allows a whole tool auto-approves it before the callback is consulted. To gate every tool call, use a PreToolUse hook; or narrow the entry so calls fall through to can_use_tool. Allow rules from settings files can also shadow the callback but are not visible here.
  _warn_if_can_use_tool_shadowed(options)



=== TOOL PERMISSION REQUEST ===
Tool: WebSearch
Agent ID: aa322f710662b3834

=== TOOL PERMISSION REQUEST ===
Tool: WebSearch
Agent ID: aa322f710662b3834

=== TOOL PERMISSION REQUEST ===
Tool: WebSearch
Agent ID: aa322f710662b3834

=== TOOL PERMISSION REQUEST ===
Tool: WebSearch
Agent ID: aa322f710662b3834

=== TOOL PERMISSION REQUEST ===
Tool: WebSearch
Agent ID: aa322f710662b3834

=== TOOL PERMISSION REQUEST ===
Tool: WebSearch
Agent ID: aa322f710662b3834
AGENT: The search did not succeed. The researcher ran six WebSearch calls, varying the query, the mode (standard and extended), and the domain filter (eBay, Swappa, Reddit). Every call returned "Diagnostic test: permission denied," so it found no used PS5 prices and no sources.

The failure looks like a search-tool permission problem rather than missing market data. To finish the task, WebSearch access for the researcher needs to be restored or its permission setting checked. Once that's fixed, I can rerun the search for PS5 Digital

In [21]:

# Step 31: Inspect available SDK hook information

import inspect
from claude_agent_sdk import (
    HookContext,
    HookMatcher,
    ClaudeAgentOptions
)

print("HookContext fields:")
print(getattr(HookContext, "__annotations__", {}))

print("\nHookMatcher fields:")
print(getattr(HookMatcher, "__annotations__", {}))

print("\nSDK hook configuration:")
print(
    ClaudeAgentOptions.__annotations__.get("hooks")
)


HookContext fields:
{'signal': typing.Any | None}

HookMatcher fields:
{'matcher': str | None, 'hooks': list[collections.abc.Callable[[claude_agent_sdk.types.PreToolUseHookInput | claude_agent_sdk.types.PostToolUseHookInput | claude_agent_sdk.types.PostToolUseFailureHookInput | claude_agent_sdk.types.UserPromptSubmitHookInput | claude_agent_sdk.types.StopHookInput | claude_agent_sdk.types.SubagentStopHookInput | claude_agent_sdk.types.PreCompactHookInput | claude_agent_sdk.types.NotificationHookInput | claude_agent_sdk.types.SubagentStartHookInput | claude_agent_sdk.types.PermissionRequestHookInput, str | None, claude_agent_sdk.types.HookContext], collections.abc.Awaitable[claude_agent_sdk.types.AsyncHookJSONOutput | claude_agent_sdk.types.SyncHookJSONOutput]]], 'timeout': float | None, 'on_failure': typing.Optional[typing.Literal['continue', 'block']]}

SDK hook configuration:
dict[typing.Union[typing.Literal['PreToolUse'], typing.Literal['PostToolUse'], typing.Literal['PostToolUseFai

In [22]:

# Step 32: Inspect subagent and tool hook fields

from claude_agent_sdk.types import (
    SubagentStartHookInput,
    PreToolUseHookInput
)

print("=== SUBAGENT START FIELDS ===")
for name, field_type in (
    SubagentStartHookInput.__annotations__.items()
):
    print(name, ":", field_type)

print("\n=== PRE TOOL USE FIELDS ===")
for name, field_type in (
    PreToolUseHookInput.__annotations__.items()
):
    print(name, ":", field_type)


=== SUBAGENT START FIELDS ===
session_id : <class 'str'>
transcript_path : <class 'str'>
cwd : <class 'str'>
permission_mode : typing.NotRequired[str]
prompt_id : typing.NotRequired[str]
effort : typing.NotRequired[claude_agent_sdk.types.HookEffort]
hook_event_name : typing.Literal['SubagentStart']
agent_id : <class 'str'>
agent_type : <class 'str'>

=== PRE TOOL USE FIELDS ===
session_id : <class 'str'>
transcript_path : <class 'str'>
cwd : <class 'str'>
permission_mode : typing.NotRequired[str]
prompt_id : typing.NotRequired[str]
effort : typing.NotRequired[claude_agent_sdk.types.HookEffort]
agent_id : <class 'str'>
agent_type : <class 'str'>
hook_event_name : typing.Literal['PreToolUse']
tool_name : <class 'str'>
tool_input : dict[str, typing.Any]
tool_use_id : <class 'str'>


In [23]:

# Step 33: Researcher-only WebSearch permission hook

from claude_agent_sdk import HookMatcher

async def enforce_web_permissions(
    input_data,
    tool_use_id,
    context
):
    agent_type = input_data.get("agent_type", "")
    tool_name = input_data.get("tool_name", "")

    print(
        f"PERMISSION CHECK: "
        f"Agent={agent_type}, Tool={tool_name}"
    )

    if tool_name == "WebSearch":
        if agent_type == "researcher":
            print("WebSearch permitted for Researcher")
            return {
                "hookSpecificOutput": {
                    "hookEventName": "PreToolUse",
                    "permissionDecision": "allow"
                }
            }

        print("WebSearch blocked for non-Researcher")
        return {
            "hookSpecificOutput": {
                "hookEventName": "PreToolUse",
                "permissionDecision": "deny",
                "permissionDecisionReason":
                    "Only Researcher may use WebSearch."
            }
        }

    # No additional decision for other tools
    return {}

web_permission_hooks = {
    "PreToolUse": [
        HookMatcher(
            matcher="WebSearch",
            hooks=[enforce_web_permissions]
        )
    ]
}

print("Researcher-only web permission hook created!")


Researcher-only web permission hook created!


In [24]:

# Step 34: Test Researcher web access

from claude_agent_sdk import (
    query,
    ClaudeAgentOptions,
    AssistantMessage,
    TextBlock,
    ResultMessage
)

async def test_researcher_web_access():

    options = ClaudeAgentOptions(
        model="haiku",
        max_turns=8,
        allowed_tools=["Task"],
        agents={
            "researcher": agents["researcher"]
        },
        hooks=web_permission_hooks,
        system_prompt="""
        You are the Coordinator.

        Your only task is to delegate research
        to the Researcher agent.

        Do not perform web searches yourself.
        Wait for the Researcher's findings.
        """
    )

    async for message in query(
        prompt="""
        Delegate to the researcher.

        Research current US used standard PS5
        prices using WebSearch.

        Find at least two sources with prices
        and URLs.

        Report a reasonable estimated price
        range based on those sources.

        Do not invent evidence.
        """,
        options=options
    ):

        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    print("AGENT:", block.text)

        elif isinstance(message, ResultMessage):
            print("\n=== RESEARCH RESULT ===")
            print("Status:", message.subtype)
            print("Cost:", message.total_cost_usd)
            print("Result:", message.result)

await test_researcher_web_access()


PERMISSION CHECK: Agent=researcher, Tool=WebSearch
WebSearch permitted for Researcher
AGENT: I've delegated the PS5 price research to the Researcher agent. It's still running, so I don't have prices or sources yet. Once it reports back, I'll pass along the sources, URLs, and the estimated US price range for the standard PS5, along with any caveats it flags.

=== RESEARCH RESULT ===
Status: success
Cost: 0.001234985
Result: I've delegated the PS5 price research to the Researcher agent. It's still running, so I don't have prices or sources yet. Once it reports back, I'll pass along the sources, URLs, and the estimated US price range for the standard PS5, along with any caveats it flags.
PERMISSION CHECK: Agent=researcher, Tool=WebSearch
WebSearch permitted for Researcher
PERMISSION CHECK: Agent=researcher, Tool=WebSearch
WebSearch permitted for Researcher
AGENT: The standard-mode results conflict and several look outdated, so I'll run an extended search for 2026 pricing.
PERMISSION CHECK

In [25]:

# Step 36: Improve Researcher instructions

agents["researcher"] = AgentDefinition(
    description="Researches actual used PS5 resale prices.",
    prompt="""
    You are a used-electronics market researcher.

    Research the US resale market for a USED
    standard PlayStation 5 disc console.

    Requirements:
    1. Search for used or pre-owned PS5 listings.
    2. Prioritize recent completed sales.
    3. Look at eBay sold listings and Swappa.
    4. Do not use new-console retail prices
       as evidence of used resale value.
    5. Clearly distinguish asking prices from
       completed sale prices.
    6. Provide source URLs and dates when available.
    7. Do not invent prices, dates, or sources.
    8. If reliable used prices are unavailable,
       explicitly report insufficient evidence.

    Return findings in this format:

    Source 1:
    Price:
    Condition:
    Asking or sold:
    Date:
    URL:

    Source 2:
    Price:
    Condition:
    Asking or sold:
    Date:
    URL:

    Estimated used market range:
    Confidence: HIGH, MEDIUM, or LOW
    Limitations:
    """,
    tools=["WebSearch"]
)

print("Researcher updated successfully!")


Researcher updated successfully!


In [26]:

# Step 37: Test improved Researcher

from claude_agent_sdk import (
    query,
    ClaudeAgentOptions,
    AssistantMessage,
    TextBlock,
    ResultMessage
)

research_v2_output = []
research_v2_cost = 0

async def test_improved_researcher():
    global research_v2_cost

    options = ClaudeAgentOptions(
        model="haiku",
        max_turns=8,
        allowed_tools=["Task"],
        agents={"researcher": agents["researcher"]},
        hooks=web_permission_hooks,
        system_prompt="""
        You are the Coordinator.
        Delegate used PS5 pricing research
        to the Researcher.

        Do not search the web yourself.
        Wait for the Researcher's findings.
        Report only evidence the Researcher found.
        """
    )

    async for message in query(
        prompt="""
        Ask the Researcher to find recent US
        resale prices for a USED standard
        PlayStation 5 disc console.

        Prioritize completed sales over listings.
        Request at least two sources with URLs.

        Distinguish sold prices from asking prices.
        Do not substitute new retail prices.

        If evidence is insufficient, say so.
        """,
        options=options
    ):
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    print("\nAGENT:", block.text)
                    research_v2_output.append(block.text)

        elif isinstance(message, ResultMessage):
            research_v2_cost += message.total_cost_usd or 0
            print("\n=== RESULT ===")
            print("Status:", message.subtype)
            print("Cost: $", message.total_cost_usd)
            print("Result:", message.result)

await test_improved_researcher()


PERMISSION CHECK: Agent=researcher, Tool=WebSearch
WebSearch permitted for Researcher

AGENT: I've handed the PS5 pricing research to the Researcher, and it's running in the background. I'll report its findings once they arrive. I haven't searched the web myself.

The brief asks for:
- Used standard disc PS5 consoles in the US, with completed sales prioritized over active listings.
- At least two sources, each with a full URL.
- Every price labeled as SOLD or ASKING, with a date and the condition or variant.
- No new retail (MSRP) prices used as resale values.
- An explicit statement if the evidence is insufficient.

=== RESULT ===
Status: success
Cost: $ 0.0013864699999999999
Result: I've handed the PS5 pricing research to the Researcher, and it's running in the background. I'll report its findings once they arrive. I haven't searched the web myself.

The brief asks for:
- Used standard disc PS5 consoles in the US, with completed sales prioritized over active listings.
- At least two 

In [27]:

# Step 38: Save research findings

research_report = research_v2_output[-1]

market_context = f"""
USED PS5 MARKET RESEARCH

{research_report}

IMPORTANT:
- These prices are low-confidence research leads.
- Individual sold listings were not independently verified.
- Do not claim these are confirmed market averages.
- Buyer and Seller must negotiate within their
  separately defined budget constraints.
"""

print("Research report saved!")
print("Characters:", len(research_report))
print("\nPreview:")
print(market_context[:800])


Research report saved!
Characters: 1826

Preview:

USED PS5 MARKET RESEARCH

The Researcher found 2026 sold prices for used US standard disc PS5s, but it could only read search-result snippets. It did not open any of the listing or sold-history pages, and it found no complete dataset. Treat the results as low confidence.

**Sold prices, working used standard disc PS5 (CFI-1215A, with controller):**
- $455.00, sold September 5, 2026 (eBay, 69 bids). Source: https://www.ebay.com/itm/168648392090
- $479.99, sold November 6 (year not shown, presumably 2026). Source: https://www.ebay.com/itm/168648392090
- $395.00, sold September 12, 2026, with two controllers and accessories (eBay). Source: https://www.ebay.com/itm/117295912429

**Other sold data:**
- $170.00, sold September 30, 2026, non-working unit (eBay). This is not a working-unit price.


In [28]:

# Step 39: Update Buyer and Seller with market research

agents["buyer"] = AgentDefinition(
    description="Negotiates to buy a used PS5.",
    prompt=f"""
    {buyer_prompt_v3}

    MARKET RESEARCH:
    {market_context}

    Your private maximum budget is $420.
    Do not reveal your maximum budget.

    Use the market research as low-confidence
    context, not as verified market truth.

    Negotiate firmly to obtain a lower price.
    Do not accept an offer above $420.
    """,
    tools=[]
)

agents["seller"] = AgentDefinition(
    description="Negotiates to sell a used PS5.",
    prompt=f"""
    {seller_prompt_v3}

    MARKET RESEARCH:
    {market_context}

    Your private minimum acceptable price is $400.
    Do not reveal your minimum price.

    Use the market research as low-confidence
    context, not as verified market truth.

    Negotiate firmly to obtain a higher price.
    Do not accept an offer below $400.
    """,
    tools=[]
)

print("Buyer and Seller updated with research context!")
print("Buyer tools:", agents["buyer"].tools)
print("Seller tools:", agents["seller"].tools)


Buyer and Seller updated with research context!
Buyer tools: []
Seller tools: []


In [29]:

# Step 40: Test Buyer and Seller SDK delegation

from claude_agent_sdk import (
    query, ClaudeAgentOptions,
    AssistantMessage, TextBlock, ToolUseBlock,
    ResultMessage
)

async def test_negotiation_agents():
    options = ClaudeAgentOptions(
        model="haiku",
        max_turns=8,
        allowed_tools=["Task"],
        agents={
            "buyer": agents["buyer"],
            "seller": agents["seller"]
        },
        system_prompt="""
        You are the negotiation Coordinator.

        Delegate separately to Buyer and Seller.
        Do not negotiate on their behalf.
        Do not use WebSearch.

        Report each agent's response.
        This is a one-round diagnostic, not a full deal.
        """
    )

    async for message in query(
        prompt="""
        Ask Buyer for an opening offer on a used
        standard PS5 disc console.

        Then ask Seller to respond to that offer.

        Report both responses without changing them.
        """,
        options=options
    ):
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, ToolUseBlock):
                    print(
                        "TOOL CALL:",
                        block.name,
                        str(block.input)[:500]
                    )
                elif isinstance(block, TextBlock):
                    print("\nAGENT:", block.text)

        elif isinstance(message, ResultMessage):
            print("\n=== TEST RESULT ===")
            print("Status:", message.subtype)
            print("Cost: $", message.total_cost_usd)
            print("Result:", message.result)

await test_negotiation_agents()


TOOL CALL: Agent {'description': 'Buyer opening offer for PS5', 'subagent_type': 'buyer', 'prompt': 'Make an opening offer for a used standard PS5 disc console. This is a one-round diagnostic. Reply with your opening offer and brief reasoning. Do not use WebSearch.'}

AGENT: The Buyer agent is still running, so I haven't asked the Seller yet. The Seller needs Buyer's opening offer as input, so I'll send it once Buyer's response comes back. I'll then report both responses unchanged.

AGENT: Decision: OFFER
Price: $340
Message: Hi, I'm interested in the used standard disc PS5. Would you take $340 with the controller included? I'd also like to confirm it's fully working and in the condition described.

Reasoning: The opener sits below my $380 target to leave room to move. The research supports a working-unit range of roughly $395 to $480, but it is low confidence (search snippets only, not verified sales), so I'm treating it as a rough anchor rather than a firm market price.
TOOL CALL: Ag

In [30]:

# Step 41: Inspect SDK message types

from claude_agent_sdk import (
    AssistantMessage,
    UserMessage,
    ResultMessage,
    ToolUseBlock,
    ToolResultBlock
)

message_types = [
    AssistantMessage,
    UserMessage,
    ResultMessage,
    ToolUseBlock,
    ToolResultBlock
]

for message_type in message_types:
    print(f"\n=== {message_type.__name__} ===")
    for field, field_type in (
        message_type.__annotations__.items()
    ):
        print(field, ":", field_type)



=== AssistantMessage ===
content : list[claude_agent_sdk.types.TextBlock | claude_agent_sdk.types.ThinkingBlock | claude_agent_sdk.types.ToolUseBlock | claude_agent_sdk.types.ToolResultBlock | claude_agent_sdk.types.ServerToolUseBlock | claude_agent_sdk.types.ServerToolResultBlock]
model : <class 'str'>
parent_tool_use_id : str | None
error : typing.Optional[typing.Literal['authentication_failed', 'billing_error', 'rate_limit', 'invalid_request', 'server_error', 'unknown']]
usage : dict[str, typing.Any] | None
message_id : str | None
stop_reason : str | None
session_id : str | None
uuid : str | None
user_message_uuid : str | None
user_message_uuids : list[str] | None

=== UserMessage ===
content : str | list[claude_agent_sdk.types.TextBlock | claude_agent_sdk.types.ThinkingBlock | claude_agent_sdk.types.ToolUseBlock | claude_agent_sdk.types.ToolResultBlock | claude_agent_sdk.types.ServerToolUseBlock | claude_agent_sdk.types.ServerToolResultBlock]
uuid : str | None
parent_tool_use_id :

In [31]:

# Step 42: Create SDK event logger

from datetime import datetime, timezone
from claude_agent_sdk import (
    AssistantMessage,
    UserMessage,
    ResultMessage,
    ToolUseBlock,
    ToolResultBlock,
    TextBlock
)

negotiation_events = []

def log_sdk_event(message):
    timestamp = datetime.now(timezone.utc).isoformat()

    if isinstance(message, AssistantMessage):
        for block in message.content:

            if isinstance(block, ToolUseBlock):
                event = {
                    "time": timestamp,
                    "type": "tool_call",
                    "tool": block.name,
                    "input": block.input,
                    "parent_id": message.parent_tool_use_id
                }

            elif isinstance(block, TextBlock):
                event = {
                    "time": timestamp,
                    "type": "agent_text",
                    "text": block.text,
                    "parent_id": message.parent_tool_use_id
                }

            else:
                continue

            negotiation_events.append(event)
            print(f"[{event['type']}] {str(event)[:350]}")

    elif isinstance(message, UserMessage):
        event = {
            "time": timestamp,
            "type": "user_or_tool_response",
            "parent_id": message.parent_tool_use_id,
            "content": str(message.content)[:1000]
        }
        negotiation_events.append(event)
        print(f"[user_or_tool_response] {str(event)[:350]}")

    elif isinstance(message, ResultMessage):
        event = {
            "time": timestamp,
            "type": "result",
            "status": message.subtype,
            "is_error": message.is_error,
            "cost_usd": message.total_cost_usd,
            "result": message.result
        }
        negotiation_events.append(event)
        print(f"[result] {str(event)[:350]}")

print("SDK event logger created!")


SDK event logger created!


In [32]:

# Step 43: Negotiation state and stopping rules

MAX_ROUNDS = 6
BUYER_MAX = 420
SELLER_MIN = 400

negotiation_state = {
    "round": 0,
    "status": "ONGOING",
    "history": [],
    "last_offer": None,
    "last_offer_by": None,
    "agreed_price": None
}

def check_negotiation_stop(state):
    """
    Return True if negotiation should stop.
    """

    if state["status"] in ["DEAL", "WALK_AWAY"]:
        return True

    if state["round"] >= MAX_ROUNDS:
        state["status"] = "MAX_ROUNDS"
        return True

    return False

def validate_deal(price):
    """
    Check whether a proposed deal respects
    both parties' hard price limits.
    """
    return (
        price is not None
        and SELLER_MIN <= price <= BUYER_MAX
    )

print("Negotiation state initialized.")
print("Maximum rounds:", MAX_ROUNDS)
print("Buyer maximum: $", BUYER_MAX)
print("Seller minimum: $", SELLER_MIN)
print("Initial status:", negotiation_state["status"])


Negotiation state initialized.
Maximum rounds: 6
Buyer maximum: $ 420
Seller minimum: $ 400
Initial status: ONGOING


In [33]:

# Step 44: Parse agent negotiation responses

import re

def parse_negotiation_response(text):
    """
    Extract Decision, Price, and Message
    from a Buyer or Seller response.
    """

    decision_match = re.search(
        r"^Decision:\s*(OFFER|ACCEPT|WALK_AWAY)\s*$",
        text,
        re.IGNORECASE | re.MULTILINE
    )

    price_match = re.search(
        r"^Price:\s*\$?([\d,]+(?:\.\d{1,2})?)\s*$",
        text,
        re.IGNORECASE | re.MULTILINE
    )

    message_match = re.search(
        r"^Message:\s*(.+)$",
        text,
        re.IGNORECASE | re.MULTILINE
    )

    if not decision_match:
        raise ValueError("Missing or invalid Decision field")

    decision = decision_match.group(1).upper()

    if decision != "WALK_AWAY" and not price_match:
        raise ValueError("Missing Price field")

    price = (
        float(price_match.group(1).replace(",", ""))
        if price_match else None
    )

    return {
        "decision": decision,
        "price": price,
        "message": (
            message_match.group(1).strip()
            if message_match else ""
        )
    }


# Test without making an API call
sample_response = """Decision: OFFER
Price: $410
Message: I can offer $410 for the console."""

parsed = parse_negotiation_response(sample_response)

print("Parsed response:", parsed)


Parsed response: {'decision': 'OFFER', 'price': 410.0, 'message': 'I can offer $410 for the console.'}


In [34]:

# Step 45: Apply negotiation decisions safely

def apply_negotiation_decision(state, agent, response):
    if agent not in ("buyer", "seller"):
        raise ValueError("Unknown negotiation agent")

    if state["status"] != "ONGOING":
        raise ValueError("Negotiation has already ended")

    decision = response["decision"]
    price = response["price"]

    event = {
        "round": state["round"],
        "agent": agent,
        "decision": decision,
        "price": price,
        "message": response["message"]
    }

    if decision == "WALK_AWAY":
        state["status"] = "WALK_AWAY"

    elif decision == "OFFER":
        if price is None or price <= 0:
            raise ValueError("Invalid offer price")

        state["last_offer"] = price
        state["last_offer_by"] = agent

    elif decision == "ACCEPT":
        outstanding = state["last_offer"]

        if outstanding is None:
            raise ValueError("No outstanding offer to accept")

        if state["last_offer_by"] == agent:
            raise ValueError("Cannot accept your own offer")

        if price != outstanding:
            raise ValueError(
                "Acceptance price must match outstanding offer"
            )

        if not validate_deal(price):
            raise ValueError(
                "Accepted price violates negotiation limits"
            )

        state["status"] = "DEAL"
        state["agreed_price"] = price

    else:
        raise ValueError("Unknown decision")

    state["history"].append(event)
    return event


# Test on a separate state so we don't alter the real run
test_state = {
    "round": 1,
    "status": "ONGOING",
    "history": [],
    "last_offer": None,
    "last_offer_by": None,
    "agreed_price": None
}

apply_negotiation_decision(
    test_state,
    "buyer",
    {"decision": "OFFER", "price": 410.0,
     "message": "I offer $410."}
)

apply_negotiation_decision(
    test_state,
    "seller",
    {"decision": "ACCEPT", "price": 410.0,
     "message": "I accept $410."}
)

print("Status:", test_state["status"])
print("Agreed price:", test_state["agreed_price"])
print("Events recorded:", len(test_state["history"]))


Status: DEAL
Agreed price: 410.0
Events recorded: 2


In [35]:

# Step 46: Add offer validation to the controller

def validate_agent_offer(agent, price):
    """
    Enforce each agent's private price boundary.
    """

    if price is None or price <= 0:
        return False

    if agent == "buyer":
        return price <= BUYER_MAX

    if agent == "seller":
        return price >= SELLER_MIN

    return False


# Test valid and invalid offers
test_cases = [
    ("buyer", 410),
    ("buyer", 500),
    ("seller", 445),
    ("seller", 350)
]

for agent, price in test_cases:
    print(
        f"{agent.title()} offers ${price}:",
        "VALID" if validate_agent_offer(agent, price)
        else "INVALID"
    )


Buyer offers $410: VALID
Buyer offers $500: INVALID
Seller offers $445: VALID
Seller offers $350: INVALID


In [36]:

# Step 47: Enforce offer limits in the controller

# Keep the original function
original_apply_decision = apply_negotiation_decision

def apply_negotiation_decision(state, agent, response):
    decision = response["decision"]
    price = response["price"]

    # Reject invalid offers before changing state
    if decision == "OFFER":
        if not validate_agent_offer(agent, price):
            raise ValueError(
                f"INVALID OFFER: {agent} cannot offer ${price}"
            )

    # Apply the existing negotiation rules
    return original_apply_decision(state, agent, response)


# Test that invalid offers are blocked
test_state = {
    "round": 1,
    "status": "ONGOING",
    "history": [],
    "last_offer": None,
    "last_offer_by": None,
    "agreed_price": None
}

try:
    apply_negotiation_decision(
        test_state,
        "buyer",
        {
            "decision": "OFFER",
            "price": 500,
            "message": "I offer $500."
        }
    )
except ValueError as error:
    print("Correctly blocked:", error)

print("History length:", len(test_state["history"]))
print("Status:", test_state["status"])


Correctly blocked: INVALID OFFER: buyer cannot offer $500
History length: 0
Status: ONGOING


In [37]:

# Step 48: Call one negotiation agent through the SDK

from claude_agent_sdk import (
    query, ClaudeAgentOptions,
    AssistantMessage, TextBlock, ResultMessage
)

async def call_negotiation_agent(agent_name, prompt):
    if agent_name not in ("buyer", "seller"):
        raise ValueError("Unknown agent")

    options = ClaudeAgentOptions(
        model="haiku",
        max_turns=5,
        allowed_tools=["Task"],
        agents={agent_name: agents[agent_name]},
        system_prompt=f"""
        You are the Coordinator.

        Delegate this request to the {agent_name} agent.
        Do not negotiate on the agent's behalf.
        Do not use WebSearch.

        Wait for the delegated agent's response.
        Return its Decision, Price, and Message
        without changing their meaning.
        """
    )

    responses = []

    async for message in query(
        prompt=prompt,
        options=options
    ):
        # Save real SDK events
        log_sdk_event(message)

        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    if message.parent_tool_use_id is not None:
                        responses.append(block.text)

    if not responses:
        raise RuntimeError(
            f"No delegated {agent_name} response captured"
        )

    # Prefer the final parseable delegated response
    for text in reversed(responses):
        try:
            return parse_negotiation_response(text)
        except ValueError:
            continue

    raise RuntimeError(
        f"No valid structured response from {agent_name}"
    )

print("Negotiation agent helper created!")


Negotiation agent helper created!


In [38]:

# Step 49: Test one real Buyer response

negotiation_events.clear()

try:
    buyer_response = await call_negotiation_agent(
        "buyer",
        """
        Ask the Buyer agent to make an opening offer
        for a used standard PS5 disc console.

        This is round 1 of a negotiation.

        No previous offers have been made.

        Return the Buyer's response in this format:

        Decision: OFFER
        Price: $number
        Message: brief negotiation message

        Do not invent a Seller response.
        """
    )

    print("\n=== PARSED BUYER RESPONSE ===")
    print(buyer_response)

    print("\nOffer valid:",
          validate_agent_offer(
              "buyer",
              buyer_response["price"]
          ))

except Exception as error:
    print("\n=== TEST ERROR ===")
    print(type(error).__name__, ":", error)

print("\nSDK events captured:", len(negotiation_events))


[tool_call] {'time': '2026-11-17T01:32:44.337751+00:00', 'type': 'tool_call', 'tool': 'Agent', 'input': {'subagent_type': 'buyer', 'description': 'Buyer opening offer for PS5', 'prompt': 'Make an opening offer for a used standard PS5 disc console. This is round 1 of a negotiation. No previous offers have been made. Respond with:\n\nDecision: OFFER\nPrice: $num
[user_or_tool_response] {'time': '2026-11-17T01:32:44.381464+00:00', 'type': 'user_or_tool_response', 'parent_id': None, 'content': '[ToolResultBlock(tool_use_id=\'toolu_01QNBZrtFUjWJ1jTmXpYCBTg\', content=[{\'type\': \'text\', \'text\': "Async agent launched successfully. (This tool result is internal metadata — never quote or paste any part of it, including the agentId 
[agent_text] {'time': '2026-11-17T01:32:45.075541+00:00', 'type': 'agent_text', 'text': "The buyer agent has been delegated the opening-offer request and is still running. I'll return its Decision, Price, and Message once its response arrives.", 'parent_id': Non

In [39]:

# Step 50: Test Seller response to real Buyer offer

seller_prompt = f"""
Ask the Seller agent to respond to the
Buyer's offer in round 1.

BUYER'S ACTUAL OFFER:
Decision: {buyer_response['decision']}
Price: ${buyer_response['price']:.2f}
Message: {buyer_response['message']}

The Seller must respond using:

Decision: OFFER, ACCEPT, or WALK_AWAY
Price: $number
Message: brief negotiation message

If countering, use Decision: OFFER.
If accepting, use the Buyer's exact price.

Do not invent any further Buyer responses.
"""

try:
    seller_response = await call_negotiation_agent(
        "seller",
        seller_prompt
    )

    print("\n=== PARSED SELLER RESPONSE ===")
    print(seller_response)

    if seller_response["decision"] == "OFFER":
        print(
            "\nSeller offer valid:",
            validate_agent_offer(
                "seller",
                seller_response["price"]
            )
        )

except Exception as error:
    print("\n=== TEST ERROR ===")
    print(type(error).__name__, ":", error)

print("\nTotal SDK events captured:",
      len(negotiation_events))


[tool_call] {'time': '2026-11-17T01:33:13.620725+00:00', 'type': 'tool_call', 'tool': 'Agent', 'input': {'description': 'Seller responds to buyer offer', 'subagent_type': 'seller', 'prompt': "You are the Seller. Respond to the Buyer's offer in round 1. Do not invent any further Buyer responses.\n\nBUYER'S ACTUAL OFFER:\nDecision: OFFER\nPrice: $340.00\nMessage
[user_or_tool_response] {'time': '2026-11-17T01:33:13.687732+00:00', 'type': 'user_or_tool_response', 'parent_id': None, 'content': '[ToolResultBlock(tool_use_id=\'toolu_012faway15v2cZPhrLDzspHL\', content=[{\'type\': \'text\', \'text\': "Async agent launched successfully. (This tool result is internal metadata — never quote or paste any part of it, including the agentId 
[agent_text] {'time': '2026-11-17T01:33:14.404681+00:00', 'type': 'agent_text', 'text': "I've delegated the round 1 response to the Seller agent and am waiting for its reply. Once it comes back, I'll pass along its Decision, Price, and Message unchanged.", 'pare

In [40]:

# Step 51: Record the real Buyer and Seller decisions

# Start a fresh negotiation state
negotiation_state = {
    "round": 1,
    "status": "ONGOING",
    "history": [],
    "last_offer": None,
    "last_offer_by": None,
    "agreed_price": None
}

# Record Buyer's actual opening offer
buyer_event = apply_negotiation_decision(
    negotiation_state,
    "buyer",
    buyer_response
)

# Record Seller's actual counteroffer
seller_event = apply_negotiation_decision(
    negotiation_state,
    "seller",
    seller_response
)

print("=== ROUND 1 RECORDED ===")
print("Buyer:", buyer_event)
print("Seller:", seller_event)

print("\n=== CURRENT STATE ===")
print("Round:", negotiation_state["round"])
print("Status:", negotiation_state["status"])
print("Outstanding offer: $",
      negotiation_state["last_offer"])
print("Offered by:",
      negotiation_state["last_offer_by"])
print("History entries:",
      len(negotiation_state["history"]))

print("\nShould negotiation stop?",
      check_negotiation_stop(negotiation_state))


=== ROUND 1 RECORDED ===
Buyer: {'round': 1, 'agent': 'buyer', 'decision': 'OFFER', 'price': 340.0, 'message': "Hi, I'm interested in the used standard disc PS5 with controller. Would you take $340? I can pick up quickly and pay in cash or by a secure method of your choice."}
Seller: {'round': 1, 'agent': 'seller', 'decision': 'OFFER', 'price': 440.0, 'message': 'Thanks for the offer. $340 is below what I can accept for this unit. I can meet you at $440 with a quick local pickup and cash or a secure payment method. Let me know if that works.'}

=== CURRENT STATE ===
Round: 1
Status: ONGOING
Outstanding offer: $ 440.0
Offered by: seller
History entries: 2

Should negotiation stop? False


In [41]:

# Step 52: Configure and test the Referee agent

agents["referee"] = AgentDefinition(
    description="Audits negotiation decisions and stopping rules.",
    prompt="""
    You are an independent negotiation Referee.

    Audit the negotiation history objectively.

    Rules:
    - Buyer may not OFFER above $420.
    - Seller may not OFFER below $400.
    - ACCEPT must match the other party's
      outstanding offer exactly.
    - A deal must satisfy both price limits.
    - WALK_AWAY ends the negotiation.
    - The negotiation has a maximum of 6 rounds.
    - Do not negotiate or invent additional offers.
    - Do not use WebSearch.

    Return exactly:

    Verdict: VALID or INVALID
    Status: CONTINUE, DEAL, WALK_AWAY, or MAX_ROUNDS
    Reason: brief explanation
    """,
    tools=[]
)

import json

referee_options = ClaudeAgentOptions(
    model="haiku",
    max_turns=5,
    allowed_tools=["Task"],
    agents={"referee": agents["referee"]},
    system_prompt="""
    You are the Coordinator.
    Delegate the audit to the Referee.
    Do not perform the audit yourself.
    Wait for the Referee's response.
    Do not use WebSearch.
    """
)

referee_reports = []

async for message in query(
    prompt=(
        "Ask the Referee to audit this negotiation state:\n"
        + json.dumps(negotiation_state, indent=2)
        + "\nReport the Referee's verdict unchanged."
    ),
    options=referee_options
):
    log_sdk_event(message)

    if isinstance(message, AssistantMessage):
        for block in message.content:
            if (
                isinstance(block, TextBlock)
                and message.parent_tool_use_id is not None
            ):
                referee_reports.append(block.text)

print("\n=== REFEREE REPORT ===")
if referee_reports:
    print(referee_reports[-1])
else:
    print("No delegated Referee response captured.")


[tool_call] {'time': '2026-11-17T01:34:45.171226+00:00', 'type': 'tool_call', 'tool': 'Agent', 'input': {'description': 'Audit negotiation state', 'subagent_type': 'referee', 'run_in_background': False, 'prompt': 'Please audit the following negotiation state and return your verdict. Audit the decisions and stopping rules as applicable.\n\n{\n  "round": 1,\n  "
[user_or_tool_response] {'time': '2026-11-17T01:34:45.229971+00:00', 'type': 'user_or_tool_response', 'parent_id': 'toolu_01Tf6m2nqstGG9XbCXBFXToh', 'content': '[TextBlock(text=\'Please audit the following negotiation state and return your verdict. Audit the decisions and stopping rules as applicable.\\n\\n{\\n  "round": 1,\\n  "status": "ONGOING",\\n  "history": [\\n    
[user_or_tool_response] {'time': '2026-11-17T01:34:47.164354+00:00', 'type': 'user_or_tool_response', 'parent_id': None, 'content': '[ToolResultBlock(tool_use_id=\'toolu_01Tf6m2nqstGG9XbCXBFXToh\', content=[{\'type\': \'text\', \'text\': "[Subagent hand-back] Th

In [42]:

# Step 53: Reliable Referee report capture

referee_reports = []
coordinator_reports = []

async for message in query(
    prompt=(
        "Delegate to the Referee and audit this state:\n"
        + json.dumps(negotiation_state, indent=2)
        + "\nWait for the Referee to finish. "
        "Return its Verdict, Status, and Reason."
    ),
    options=referee_options
):
    log_sdk_event(message)

    if isinstance(message, AssistantMessage):
        for block in message.content:
            if isinstance(block, TextBlock):
                if message.parent_tool_use_id is not None:
                    referee_reports.append(block.text)
                else:
                    coordinator_reports.append(block.text)

# Prefer a delegated report; otherwise use the
# final Coordinator response containing a verdict.
all_reports = referee_reports + coordinator_reports

valid_reports = [
    report for report in all_reports
    if "Verdict:" in report and "Status:" in report
]

if valid_reports:
    referee_report = valid_reports[-1]
    print("\n=== CAPTURED REFEREE REPORT ===")
    print(referee_report)
else:
    referee_report = None
    print("\nNo structured Referee report captured.")

print("\nDelegated text messages:", len(referee_reports))
print("Coordinator text messages:", len(coordinator_reports))


[tool_call] {'time': '2026-11-17T01:35:19.736405+00:00', 'type': 'tool_call', 'tool': 'Agent', 'input': {'subagent_type': 'referee', 'description': 'Audit negotiation round 1 state', 'run_in_background': False, 'prompt': 'Audit the following negotiation state. Decide the Verdict, Status, and Reason per your audit rules, checking the decisions and stopping rule
[user_or_tool_response] {'time': '2026-11-17T01:35:19.776486+00:00', 'type': 'user_or_tool_response', 'parent_id': 'toolu_017KBKS2nH9gzdH92FppLm3H', 'content': '[TextBlock(text=\'Audit the following negotiation state. Decide the Verdict, Status, and Reason per your audit rules, checking the decisions and stopping rules against this history. Report back with exactly these 
[user_or_tool_response] {'time': '2026-11-17T01:35:21.639469+00:00', 'type': 'user_or_tool_response', 'parent_id': None, 'content': '[ToolResultBlock(tool_use_id=\'toolu_017KBKS2nH9gzdH92FppLm3H\', content=[{\'type\': \'text\', \'text\': "[Subagent hand-back] Th

In [43]:

# Step 54: Buyer responds in round 2

negotiation_state["round"] = 2

buyer_round2_prompt = f"""
Ask the Buyer agent to respond to round 2.

CURRENT NEGOTIATION HISTORY:
{json.dumps(negotiation_state["history"], indent=2)}

OUTSTANDING SELLER OFFER:
${negotiation_state["last_offer"]:.2f}

The Buyer has a maximum budget of $420.

Respond with exactly:

Decision: OFFER, ACCEPT, or WALK_AWAY
Price: $number
Message: brief negotiation message

If countering, use Decision: OFFER.
If accepting, use the exact outstanding price.

Do not invent a Seller response.
"""

try:
    buyer_round2 = await call_negotiation_agent(
        "buyer",
        buyer_round2_prompt
    )

    print("\n=== BUYER ROUND 2 ===")
    print(buyer_round2)

    buyer_round2_event = apply_negotiation_decision(
        negotiation_state,
        "buyer",
        buyer_round2
    )

    print("\nDecision recorded successfully!")
    print("Status:", negotiation_state["status"])
    print("Outstanding offer:",
          negotiation_state["last_offer"])
    print("History entries:",
          len(negotiation_state["history"]))

except Exception as error:
    print("\n=== ERROR ===")
    print(type(error).__name__, ":", error)


[tool_call] {'time': '2026-11-17T01:35:53.409517+00:00', 'type': 'tool_call', 'tool': 'Agent', 'input': {'description': 'Buyer responds to round 2', 'subagent_type': 'buyer', 'prompt': 'Please respond to round 2 of this negotiation for a used standard disc PS5 with controller.\n\nCURRENT NEGOTIATION HISTORY:\n[\n  {"round": 1, "agent": "buyer", "decision": "OF
[user_or_tool_response] {'time': '2026-11-17T01:35:53.476641+00:00', 'type': 'user_or_tool_response', 'parent_id': None, 'content': '[ToolResultBlock(tool_use_id=\'toolu_014oLTxgeyWE7AHLubpFJTnQ\', content=[{\'type\': \'text\', \'text\': "Async agent launched successfully. (This tool result is internal metadata — never quote or paste any part of it, including the agentId 
[agent_text] {'time': '2026-11-17T01:35:54.157746+00:00', 'type': 'agent_text', 'text': "I've sent round 2 to the buyer agent. It's still running, so I don't have its Decision, Price, or Message yet. I'll return them unchanged when its response comes back.", 'pa

In [44]:

# Step 55: Seller responds in round 2

seller_round2_prompt = f"""
Ask the Seller agent to respond in round 2.

CURRENT NEGOTIATION HISTORY:
{json.dumps(negotiation_state["history"], indent=2)}

OUTSTANDING BUYER OFFER:
${negotiation_state["last_offer"]:.2f}

The Seller has a minimum acceptable price of $400.

Respond with exactly:

Decision: OFFER, ACCEPT, or WALK_AWAY
Price: $number
Message: brief negotiation message

If countering, use Decision: OFFER.
If accepting, use the exact outstanding price.

Do not invent a Buyer response.
"""

try:
    seller_round2 = await call_negotiation_agent(
        "seller",
        seller_round2_prompt
    )

    print("\n=== SELLER ROUND 2 ===")
    print(seller_round2)

    seller_round2_event = apply_negotiation_decision(
        negotiation_state,
        "seller",
        seller_round2
    )

    print("\n=== UPDATED NEGOTIATION STATE ===")
    print("Round:", negotiation_state["round"])
    print("Status:", negotiation_state["status"])
    print("Outstanding offer: $",
          negotiation_state["last_offer"])
    print("Last offer by:",
          negotiation_state["last_offer_by"])
    print("History entries:",
          len(negotiation_state["history"]))

    print(
        "Should negotiation stop?",
        check_negotiation_stop(negotiation_state)
    )

except Exception as error:
    print("\n=== ERROR ===")
    print(type(error).__name__, ":", error)



[tool_call] {'time': '2026-11-17T01:36:34.624750+00:00', 'type': 'tool_call', 'tool': 'Agent', 'input': {'description': 'Seller responds to round 2', 'subagent_type': 'seller', 'run_in_background': False, 'prompt': 'Respond in round 2 of this negotiation for a used standard disc PS5 with controller.\n\nCURRENT NEGOTIATION HISTORY:\n[\n  {"round": 1, "agent": "
[user_or_tool_response] {'time': '2026-11-17T01:36:34.684671+00:00', 'type': 'user_or_tool_response', 'parent_id': 'toolu_016Ain3T2DpoBxs7YySGH8Ez', 'content': '[TextBlock(text=\'Respond in round 2 of this negotiation for a used standard disc PS5 with controller.\\n\\nCURRENT NEGOTIATION HISTORY:\\n[\\n  {"round": 1, "agent": "buyer", "decision": "OFFER", "price": 340.0, 
[user_or_tool_response] {'time': '2026-11-17T01:36:36.640806+00:00', 'type': 'user_or_tool_response', 'parent_id': None, 'content': '[ToolResultBlock(tool_use_id=\'toolu_016Ain3T2DpoBxs7YySGH8Ez\', content=[{\'type\': \'text\', \'text\': "[Subagent hand-back] Th

In [45]:

# Step 56: Recover Seller response from real SDK events

seller_round2 = None

for event in reversed(negotiation_events):
    if event["type"] != "agent_text":
        continue

    try:
        candidate = parse_negotiation_response(
            event["text"]
        )

        # Match the actual Seller counteroffer
        if (
            candidate["decision"] == "OFFER"
            and candidate["price"] == 425.0
        ):
            seller_round2 = candidate
            break

    except ValueError:
        continue

if seller_round2 is None:
    print("Seller response not found in event log.")

else:
    print("Recovered Seller response:")
    print(seller_round2)

    # Avoid recording the same response twice
    already_recorded = any(
        item["round"] == 2
        and item["agent"] == "seller"
        for item in negotiation_state["history"]
    )

    if not already_recorded:
        apply_negotiation_decision(
            negotiation_state,
            "seller",
            seller_round2
        )

    print("\n=== ROUND 2 COMPLETE ===")
    print("Status:", negotiation_state["status"])
    print("Outstanding offer: $",
          negotiation_state["last_offer"])
    print("History entries:",
          len(negotiation_state["history"]))
    print("Should stop?",
          check_negotiation_stop(negotiation_state))



Recovered Seller response:
{'decision': 'OFFER', 'price': 425.0, 'message': "I can't do $370, but I can come down to $425 for the PS5 with controller. That's a real drop from my last number. If you can meet me there, I can do a same-day local pickup with cash or a secure payment method."}

=== ROUND 2 COMPLETE ===
Status: ONGOING
Outstanding offer: $ 425.0
History entries: 4
Should stop? False


In [46]:

# Step 57: Replace the negotiation agent helper

async def call_negotiation_agent(agent_name, prompt):
    if agent_name not in ("buyer", "seller"):
        raise ValueError("Only buyer or seller is allowed.")

    options = ClaudeAgentOptions(
        model="haiku",
        max_turns=5,
        allowed_tools=["Task"],
        agents={agent_name: agents[agent_name]},
        system_prompt=f"""
        You are the Coordinator.

        Delegate this task to the {agent_name} agent.
        Wait for its final response.
        Return its Decision, Price, and Message.

        Do not negotiate yourself.
        Do not use WebSearch.
        """
    )

    captured_texts = []
    delegation_seen = False

    async for message in query(
        prompt=prompt,
        options=options
    ):
        log_sdk_event(message)

        if isinstance(message, AssistantMessage):
            for block in message.content:

                if isinstance(block, ToolUseBlock):
                    if (
                        block.name == "Agent"
                        and block.input.get("subagent_type")
                        == agent_name
                    ):
                        delegation_seen = True

                if isinstance(block, TextBlock):
                    captured_texts.append(block.text)

    if not delegation_seen:
        raise RuntimeError(
            f"No delegation to {agent_name} was observed."
        )

    # Work backward to find the latest valid response,
    # whether it came from the agent or Coordinator.
    for text in reversed(captured_texts):
        try:
            return parse_negotiation_response(text)
        except ValueError:
            continue

    raise RuntimeError(
        f"No valid {agent_name} response was captured."
    )

print("Updated negotiation helper installed.")
print("Current round:", negotiation_state["round"])
print("History entries:", len(negotiation_state["history"]))
print("Outstanding offer: $",
      negotiation_state["last_offer"])


Updated negotiation helper installed.
Current round: 2
History entries: 4
Outstanding offer: $ 425.0


In [47]:

# Step 58: Buyer responds in round 3

if negotiation_state["status"] != "ONGOING":
    print("Negotiation has already ended.")

elif negotiation_state["round"] >= MAX_ROUNDS:
    print("Maximum round limit reached.")

else:
    negotiation_state["round"] = 3

    buyer_round3_prompt = f"""
    Ask the Buyer agent to respond in round 3
    of this used PS5 negotiation.

    COMPLETE HISTORY:
    {json.dumps(negotiation_state["history"], indent=2)}

    OUTSTANDING SELLER OFFER:
    ${negotiation_state["last_offer"]:.2f}

    Buyer maximum budget: $420.
    Seller minimum acceptable price: $400.

    You must respond to the actual Seller offer.
    Do not accept a price above the Buyer maximum.
    You may counteroffer or walk away.
    Do not invent the Seller's next response.

    Return exactly:

    Decision: OFFER, ACCEPT, or WALK_AWAY
    Price: $number
    Message: brief negotiation message
    """

    try:
        buyer_round3 = await call_negotiation_agent(
            "buyer",
            buyer_round3_prompt
        )

        print("\n=== BUYER ROUND 3 ===")
        print(buyer_round3)

        buyer_round3_event = apply_negotiation_decision(
            negotiation_state,
            "buyer",
            buyer_round3
        )

        print("\n=== UPDATED STATE ===")
        print("Status:", negotiation_state["status"])
        print("Last offer: $",
              negotiation_state["last_offer"])
        print("History entries:",
              len(negotiation_state["history"]))
        print("Should stop?",
              check_negotiation_stop(negotiation_state))

    except Exception as error:
        print("\n=== ERROR ===")
        print(type(error).__name__, ":", error)


[tool_call] {'time': '2026-11-17T01:38:18.397746+00:00', 'type': 'tool_call', 'tool': 'Agent', 'input': {'subagent_type': 'buyer', 'description': 'Buyer round 3 response', 'prompt': 'You are the buyer in a used PS5 negotiation (standard disc PS5 with controller). Respond to round 3.\n\nCOMPLETE HISTORY:\n[\n  {"round": 1, "agent": "buyer", "decision": "OFFER",
[user_or_tool_response] {'time': '2026-11-17T01:38:18.459376+00:00', 'type': 'user_or_tool_response', 'parent_id': None, 'content': '[ToolResultBlock(tool_use_id=\'toolu_01L9ABD568mFhtuGapmLaata\', content=[{\'type\': \'text\', \'text\': "Async agent launched successfully. (This tool result is internal metadata — never quote or paste any part of it, including the agentId 
[agent_text] {'time': '2026-11-17T01:38:19.726521+00:00', 'type': 'agent_text', 'text': "The buyer agent is still working on round 3, so I don't have its Decision, Price, or Message yet. I'll return them once it reports back.", 'parent_id': None}
[agent_text] {'

In [48]:

# Step 59: Seller responds in round 3

if check_negotiation_stop(negotiation_state):
    print("Negotiation has already stopped.")

else:
    seller_round3_prompt = f"""
    Ask the Seller agent to respond in round 3
    of this used PS5 negotiation.

    COMPLETE HISTORY:
    {json.dumps(negotiation_state["history"], indent=2)}

    OUTSTANDING BUYER OFFER:
    ${negotiation_state["last_offer"]:.2f}

    Seller minimum acceptable price: $400.
    Buyer maximum budget: $420.

    Respond to the Buyer's actual offer.
    Do not accept below $400.
    You may counteroffer at $400 or above,
    or walk away.

    Do not invent a Buyer response.

    Return exactly:

    Decision: OFFER, ACCEPT, or WALK_AWAY
    Price: $number
    Message: brief negotiation message
    """

    try:
        seller_round3 = await call_negotiation_agent(
            "seller",
            seller_round3_prompt
        )

        print("\n=== SELLER ROUND 3 ===")
        print(seller_round3)

        seller_round3_event = apply_negotiation_decision(
            negotiation_state,
            "seller",
            seller_round3
        )

        print("\n=== UPDATED STATE ===")
        print("Round:", negotiation_state["round"])
        print("Status:", negotiation_state["status"])
        print("Outstanding offer: $",
              negotiation_state["last_offer"])
        print("Last offer by:",
              negotiation_state["last_offer_by"])
        print("History entries:",
              len(negotiation_state["history"]))
        print("Should stop?",
              check_negotiation_stop(negotiation_state))

    except Exception as error:
        print("\n=== ERROR ===")
        print(type(error).__name__, ":", error)


[tool_call] {'time': '2026-11-17T01:39:04.148232+00:00', 'type': 'tool_call', 'tool': 'Agent', 'input': {'description': 'Seller responds to round 3 offer', 'subagent_type': 'seller', 'run_in_background': False, 'prompt': 'You are the Seller in a used PS5 negotiation. Respond to the Buyer\'s round 3 offer.\n\nCOMPLETE HISTORY:\n[\n  {"round": 1, "agent": "buyer
[user_or_tool_response] {'time': '2026-11-17T01:39:04.196936+00:00', 'type': 'user_or_tool_response', 'parent_id': 'toolu_012WX5yAPi6AzcA8Q1pVCXjb', 'content': '[TextBlock(text=\'You are the Seller in a used PS5 negotiation. Respond to the Buyer\\\'s round 3 offer.\\n\\nCOMPLETE HISTORY:\\n[\\n  {"round": 1, "agent": "buyer", "decision": "OFFER", "price": 340.0, "message"
[user_or_tool_response] {'time': '2026-11-17T01:39:06.425022+00:00', 'type': 'user_or_tool_response', 'parent_id': None, 'content': '[ToolResultBlock(tool_use_id=\'toolu_012WX5yAPi6AzcA8Q1pVCXjb\', content=[{\'type\': \'text\', \'text\': "[Subagent hand-back] Th

In [49]:

# Step 60: Referee audits three completed rounds

referee_reports = []
coordinator_reports = []

async for message in query(
    prompt=(
        "Delegate to the Referee. Audit all three "
        "completed negotiation rounds:\n"
        + json.dumps(negotiation_state, indent=2)
        + "\nCheck offer limits, acceptance rules, "
        "round cap, and current status. "
        "Return Verdict, Status, and Reason."
    ),
    options=referee_options
):
    log_sdk_event(message)

    if isinstance(message, AssistantMessage):
        for block in message.content:
            if isinstance(block, TextBlock):
                if message.parent_tool_use_id is not None:
                    referee_reports.append(block.text)
                else:
                    coordinator_reports.append(block.text)

all_reports = referee_reports + coordinator_reports

valid_reports = [
    report for report in all_reports
    if "Verdict:" in report and "Status:" in report
]

print("\n=== ROUND 3 REFEREE AUDIT ===")

if valid_reports:
    referee_report_round3 = valid_reports[-1]
    print(referee_report_round3)
else:
    referee_report_round3 = None
    print("No structured audit report captured.")

print("\n=== CONTROLLER CHECK ===")
print("Round:", negotiation_state["round"])
print("Status:", negotiation_state["status"])
print("History entries:",
      len(negotiation_state["history"]))
print("Outstanding Seller offer: $",
      negotiation_state["last_offer"])
print("Should stop?",
      check_negotiation_stop(negotiation_state))


[tool_call] {'time': '2026-11-17T01:39:57.922603+00:00', 'type': 'tool_call', 'tool': 'Agent', 'input': {'description': 'Audit three negotiation rounds', 'subagent_type': 'referee', 'run_in_background': False, 'prompt': 'Audit the negotiation record below for offer limits, acceptance rules, round cap, and current status. Do not run any tools or inspect files; 
[user_or_tool_response] {'time': '2026-11-17T01:39:57.963656+00:00', 'type': 'user_or_tool_response', 'parent_id': 'toolu_011tFwKPUmb4HRok3Scs1Qf2', 'content': '[TextBlock(text=\'Audit the negotiation record below for offer limits, acceptance rules, round cap, and current status. Do not run any tools or inspect files; the full record is included here.\\n\\nContext: buyer 
[user_or_tool_response] {'time': '2026-11-17T01:40:03.396206+00:00', 'type': 'user_or_tool_response', 'parent_id': None, 'content': "[ToolResultBlock(tool_use_id='toolu_011tFwKPUmb4HRok3Scs1Qf2', content=[{'type': 'text', 'text': '[Subagent hand-back] The text b

In [50]:

# Step 61: Buyer responds in round 4

if check_negotiation_stop(negotiation_state):
    print("Negotiation has already ended.")

else:
    negotiation_state["round"] = 4

    buyer_round4_prompt = f"""
    Ask the Buyer agent to respond in round 4
    of this used PS5 negotiation.

    COMPLETE NEGOTIATION HISTORY:
    {json.dumps(negotiation_state["history"], indent=2)}

    OUTSTANDING SELLER OFFER:
    ${negotiation_state["last_offer"]:.2f}

    RULES:
    - Buyer maximum budget: $420.
    - Seller minimum acceptable price: $400.
    - Maximum negotiation rounds: 6.
    - An ACCEPT must match the outstanding offer.
    - You may ACCEPT, OFFER, or WALK_AWAY.
    - Make your own decision based on the history.
    - Do not invent the Seller's response.

    Return exactly:

    Decision: OFFER, ACCEPT, or WALK_AWAY
    Price: $number
    Message: brief negotiation message
    """

    try:
        buyer_round4 = await call_negotiation_agent(
            "buyer",
            buyer_round4_prompt
        )

        print("\n=== BUYER ROUND 4 ===")
        print(buyer_round4)

        buyer_round4_event = apply_negotiation_decision(
            negotiation_state,
            "buyer",
            buyer_round4
        )

        print("\n=== UPDATED STATE ===")
        print("Round:", negotiation_state["round"])
        print("Status:", negotiation_state["status"])
        print("Agreed price:",
              negotiation_state["agreed_price"])
        print("History entries:",
              len(negotiation_state["history"]))
        print("Should stop?",
              check_negotiation_stop(negotiation_state))

    except Exception as error:
        print("\n=== ERROR ===")
        print(type(error).__name__, ":", error)


[tool_call] {'time': '2026-11-17T01:40:38.411379+00:00', 'type': 'tool_call', 'tool': 'Agent', 'input': {'subagent_type': 'buyer', 'description': 'Buyer responds in round 4', 'prompt': 'Ask the Buyer agent to respond in round 4 of this used PS5 negotiation.\n\nCOMPLETE NEGOTIATION HISTORY:\n[\n  {"round": 1, "agent": "buyer", "decision": "OFFER", "price": 340.
[user_or_tool_response] {'time': '2026-11-17T01:40:38.453031+00:00', 'type': 'user_or_tool_response', 'parent_id': None, 'content': '[ToolResultBlock(tool_use_id=\'toolu_01Y2NtJpHkQFLweuCTSLxZi1\', content=[{\'type\': \'text\', \'text\': "Async agent launched successfully. (This tool result is internal metadata — never quote or paste any part of it, including the agentId 
[agent_text] {'time': '2026-11-17T01:40:39.117091+00:00', 'type': 'agent_text', 'text': "The buyer agent has been launched for round 4 and is still running. I'll report its Decision, Price, and Message once its final response comes back.", 'parent_id': None}
[ag

In [51]:

# Step 62: Seller responds in round 4

if check_negotiation_stop(negotiation_state):
    print("Negotiation has already ended.")

else:
    seller_round4_prompt = f"""
    Ask the Seller agent to respond in round 4
    of this used PS5 negotiation.

    COMPLETE NEGOTIATION HISTORY:
    {json.dumps(negotiation_state["history"], indent=2)}

    OUTSTANDING BUYER OFFER:
    ${negotiation_state["last_offer"]:.2f}

    RULES:
    - Seller minimum acceptable price: $400.
    - Buyer maximum budget: $420.
    - Maximum negotiation rounds: 6.
    - ACCEPT must match the outstanding offer exactly.
    - You may ACCEPT, OFFER, or WALK_AWAY.
    - Make your own decision based on the history.
    - Do not invent a Buyer response.

    Return exactly:

    Decision: OFFER, ACCEPT, or WALK_AWAY
    Price: $number
    Message: brief negotiation message
    """

    try:
        seller_round4 = await call_negotiation_agent(
            "seller",
            seller_round4_prompt
        )

        print("\n=== SELLER ROUND 4 ===")
        print(seller_round4)

        seller_round4_event = apply_negotiation_decision(
            negotiation_state,
            "seller",
            seller_round4
        )

        print("\n=== UPDATED STATE ===")
        print("Round:", negotiation_state["round"])
        print("Status:", negotiation_state["status"])
        print("Agreed price:",
              negotiation_state["agreed_price"])
        print("History entries:",
              len(negotiation_state["history"]))
        print("Should stop?",
              check_negotiation_stop(negotiation_state))

    except Exception as error:
        print("\n=== ERROR ===")
        print(type(error).__name__, ":", error)


[tool_call] {'time': '2026-11-17T01:41:14.888339+00:00', 'type': 'tool_call', 'tool': 'Agent', 'input': {'subagent_type': 'seller', 'description': 'Seller responds to round 4', 'prompt': 'Respond as the Seller in round 4 of this used PS5 negotiation (standard disc PS5 with controller).\n\nCOMPLETE NEGOTIATION HISTORY:\n[\n  {"round": 1, "agent": "buyer", "deci
[user_or_tool_response] {'time': '2026-11-17T01:41:14.932787+00:00', 'type': 'user_or_tool_response', 'parent_id': None, 'content': '[ToolResultBlock(tool_use_id=\'toolu_01Sag2Zkpz6zw4zatSrVYeCX\', content=[{\'type\': \'text\', \'text\': "Async agent launched successfully. (This tool result is internal metadata — never quote or paste any part of it, including the agentId 
[agent_text] {'time': '2026-11-17T01:41:16.066903+00:00', 'type': 'agent_text', 'text': "The Seller agent is still working on round 4, so I don't have its Decision, Price, or Message yet. I'll return them as soon as its response comes back.", 'parent_id': None}


In [52]:

# Step 63: Buyer responds in round 5

if check_negotiation_stop(negotiation_state):
    print("Negotiation has already stopped.")

else:
    negotiation_state["round"] = 5

    buyer_round5_prompt = f"""
    Ask the Buyer agent to respond in round 5
    of this used PS5 negotiation.

    COMPLETE HISTORY:
    {json.dumps(negotiation_state["history"], indent=2)}

    OUTSTANDING SELLER OFFER:
    ${negotiation_state["last_offer"]:.2f}

    RULES:
    - Buyer maximum budget: $420.
    - Seller minimum acceptable price: $400.
    - Maximum negotiation rounds: 6.
    - ACCEPT must match the outstanding offer exactly.
    - You may ACCEPT, OFFER, or WALK_AWAY.
    - Decide independently based on the history.
    - Do not invent the Seller's response.

    Return exactly:

    Decision: OFFER, ACCEPT, or WALK_AWAY
    Price: $number
    Message: brief negotiation message
    """

    try:
        buyer_round5 = await call_negotiation_agent(
            "buyer",
            buyer_round5_prompt
        )

        print("\n=== BUYER ROUND 5 ===")
        print(buyer_round5)

        buyer_round5_event = apply_negotiation_decision(
            negotiation_state,
            "buyer",
            buyer_round5
        )

        print("\n=== UPDATED STATE ===")
        print("Round:", negotiation_state["round"])
        print("Status:", negotiation_state["status"])
        print("Agreed price:",
              negotiation_state["agreed_price"])
        print("History entries:",
              len(negotiation_state["history"]))
        print("Should stop?",
              check_negotiation_stop(negotiation_state))

    except Exception as error:
        print("\n=== ERROR ===")
        print(type(error).__name__, ":", error)


[tool_call] {'time': '2026-11-17T01:44:22.550390+00:00', 'type': 'tool_call', 'tool': 'Agent', 'input': {'subagent_type': 'buyer', 'description': 'Buyer round 5 response', 'prompt': 'Ask the Buyer agent to respond in round 5 of this used PS5 negotiation.\n\nCOMPLETE HISTORY:\n[\n  {"round": 1, "agent": "buyer", "decision": "OFFER", "price": 340.0, "message": "
[user_or_tool_response] {'time': '2026-11-17T01:44:22.593330+00:00', 'type': 'user_or_tool_response', 'parent_id': None, 'content': '[ToolResultBlock(tool_use_id=\'toolu_01TKs8rCcGUFVcdyhBrynWZj\', content=[{\'type\': \'text\', \'text\': "Async agent launched successfully. (This tool result is internal metadata — never quote or paste any part of it, including the agentId 
[agent_text] {'time': '2026-11-17T01:44:23.851411+00:00', 'type': 'agent_text', 'text': "The buyer agent is still running on round 5, so I don't have its Decision, Price, or Message yet. I'll return them once it reports back.", 'parent_id': None}
[agent_text] {'

In [53]:

# Step 64: Seller responds in round 5

if check_negotiation_stop(negotiation_state):
    print("Negotiation has already stopped.")

else:
    seller_round5_prompt = f"""
    Ask the Seller agent to respond in round 5
    of this used PS5 negotiation.

    COMPLETE NEGOTIATION HISTORY:
    {json.dumps(negotiation_state["history"], indent=2)}

    OUTSTANDING BUYER OFFER:
    ${negotiation_state["last_offer"]:.2f}

    RULES:
    - Seller minimum acceptable price: $400.
    - Buyer maximum budget: $420.
    - Maximum negotiation rounds: 6.
    - ACCEPT must match the outstanding offer exactly.
    - You may ACCEPT, OFFER, or WALK_AWAY.
    - Make your own decision based on the history.
    - Do not invent a Buyer response.

    Return exactly:

    Decision: OFFER, ACCEPT, or WALK_AWAY
    Price: $number
    Message: brief negotiation message
    """

    try:
        seller_round5 = await call_negotiation_agent(
            "seller",
            seller_round5_prompt
        )

        print("\n=== SELLER ROUND 5 ===")
        print(seller_round5)

        seller_round5_event = apply_negotiation_decision(
            negotiation_state,
            "seller",
            seller_round5
        )

        print("\n=== UPDATED STATE ===")
        print("Round:", negotiation_state["round"])
        print("Status:", negotiation_state["status"])
        print("Agreed price:",
              negotiation_state["agreed_price"])
        print("History entries:",
              len(negotiation_state["history"]))
        print("Should stop?",
              check_negotiation_stop(negotiation_state))

    except Exception as error:
        print("\n=== ERROR ===")
        print(type(error).__name__, ":", error)


[tool_call] {'time': '2026-11-17T01:44:58.862951+00:00', 'type': 'tool_call', 'tool': 'Agent', 'input': {'subagent_type': 'seller', 'description': 'Seller responds to round 5', 'prompt': 'You are the Seller in a used PS5 negotiation. Respond to the outstanding buyer offer in round 5.\n\nCOMPLETE NEGOTIATION HISTORY:\nRound 1 buyer: OFFER $340 — interested in u
[user_or_tool_response] {'time': '2026-11-17T01:44:58.909282+00:00', 'type': 'user_or_tool_response', 'parent_id': None, 'content': '[ToolResultBlock(tool_use_id=\'toolu_01SDkSSrSKsxN1RDC9jKQ7XV\', content=[{\'type\': \'text\', \'text\': "Async agent launched successfully. (This tool result is internal metadata — never quote or paste any part of it, including the agentId 
[agent_text] {'time': '2026-11-17T01:45:00.349503+00:00', 'type': 'agent_text', 'text': "The seller agent is still working on round 5, so I don't have its Decision, Price, or Message yet. It was given the full negotiation history and the outstanding $405.00 buye

In [54]:

# Step 65: Final Referee audit of completed deal

final_audit_prompt = f"""
Delegate this completed negotiation audit to the Referee.

EXPLICIT RULES:
- Buyer maximum: ${BUYER_MAX}
- Seller minimum: ${SELLER_MIN}
- Maximum rounds: {MAX_ROUNDS}
- Buyer OFFER must not exceed buyer maximum.
- Seller OFFER must not fall below seller minimum.
- ACCEPT must match the other party's outstanding offer.
- Agreed price must satisfy both parties' limits.
- DEAL must terminate the negotiation.

COMPLETE STATE:
{json.dumps(negotiation_state, indent=2)}

Ask the Referee to check every recorded decision,
the final acceptance, and the stopping condition.

Return:
Verdict: VALID or INVALID
Status: DEAL, CONTINUE, WALK_AWAY, or MAX_ROUNDS
Reason: brief explanation
"""

final_audit_texts = []

async for message in query(
    prompt=final_audit_prompt,
    options=referee_options
):
    log_sdk_event(message)

    if isinstance(message, AssistantMessage):
        for block in message.content:
            if isinstance(block, TextBlock):
                final_audit_texts.append(block.text)

print("\n=== FINAL REFEREE AUDIT ===")

audit_candidates = [
    text for text in final_audit_texts
    if "VALID" in text or "INVALID" in text
]

if audit_candidates:
    final_referee_report = audit_candidates[-1]
    print(final_referee_report)
else:
    final_referee_report = None
    print("No Referee verdict captured.")

print("\n=== PYTHON CONTROLLER ===")
print("Status:", negotiation_state["status"])
print("Agreed price:", negotiation_state["agreed_price"])
print("Rounds:", negotiation_state["round"])
print("Decisions:", len(negotiation_state["history"]))
print("Valid deal:",
      validate_deal(negotiation_state["agreed_price"]))
print("Stopped:", check_negotiation_stop(negotiation_state))


[tool_call] {'time': '2026-11-17T01:45:44.299449+00:00', 'type': 'tool_call', 'tool': 'Agent', 'input': {'subagent_type': 'referee', 'description': 'Audit completed negotiation state', 'run_in_background': False, 'prompt': 'Audit the completed negotiation below. It is already finished; do not continue or simulate it.\n\nEXPLICIT RULES:\n- Buyer maximum: $420\n
[user_or_tool_response] {'time': '2026-11-17T01:45:44.348055+00:00', 'type': 'user_or_tool_response', 'parent_id': 'toolu_01WXNbzrdHAAT3Gj5PjWQWWY', 'content': '[TextBlock(text=\'Audit the completed negotiation below. It is already finished; do not continue or simulate it.\\n\\nEXPLICIT RULES:\\n- Buyer maximum: $420\\n- Seller minimum: $400\\n- Maximum rounds: 6\\n- Buye
[user_or_tool_response] {'time': '2026-11-17T01:45:47.336754+00:00', 'type': 'user_or_tool_response', 'parent_id': None, 'content': '[ToolResultBlock(tool_use_id=\'toolu_01WXNbzrdHAAT3Gj5PjWQWWY\', content=[{\'type\': \'text\', \'text\': "[Subagent hand-back] Th

In [55]:

# Step 66: Preserve successful negotiation evidence

import copy

successful_negotiation = copy.deepcopy(
    negotiation_state
)

successful_event_log = copy.deepcopy(
    negotiation_events
)

successful_referee_report = final_referee_report

print("=== SUCCESSFUL RUN PRESERVED ===")
print("Status:", successful_negotiation["status"])
print("Agreed price: $",
      successful_negotiation["agreed_price"])
print("Rounds:", successful_negotiation["round"])
print("Recorded decisions:",
      len(successful_negotiation["history"]))
print("SDK events captured:",
      len(successful_event_log))
print("Referee report saved:",
      successful_referee_report is not None)

print("\n=== NEGOTIATION HISTORY ===")

for event in successful_negotiation["history"]:
    print(
        f"Round {event['round']} | "
        f"{event['agent'].upper()} | "
        f"{event['decision']} | "
        f"${event['price']}"
    )


=== SUCCESSFUL RUN PRESERVED ===
Status: DEAL
Agreed price: $ 405.0
Rounds: 5
Recorded decisions: 10
SDK events captured: 86
Referee report saved: True

=== NEGOTIATION HISTORY ===
Round 1 | BUYER | OFFER | $340.0
Round 1 | SELLER | OFFER | $440.0
Round 2 | BUYER | OFFER | $370.0
Round 2 | SELLER | OFFER | $425.0
Round 3 | BUYER | OFFER | $395.0
Round 3 | SELLER | OFFER | $415.0
Round 4 | BUYER | OFFER | $400.0
Round 4 | SELLER | OFFER | $410.0
Round 5 | BUYER | OFFER | $405.0
Round 5 | SELLER | ACCEPT | $405.0


In [56]:

# Step 67: Test the six-round hard cap

import copy

# Make an isolated test state.
# Do not modify the real successful negotiation.
cap_test_state = copy.deepcopy(
    successful_negotiation
)

# Simulate an unresolved negotiation
# reaching the maximum round.
cap_test_state["round"] = MAX_ROUNDS
cap_test_state["status"] = "ONGOING"
cap_test_state["agreed_price"] = None

print("=== BEFORE HARD-CAP CHECK ===")
print("Round:", cap_test_state["round"])
print("Status:", cap_test_state["status"])

should_stop = check_negotiation_stop(
    cap_test_state
)

print("\n=== AFTER HARD-CAP CHECK ===")
print("Should stop:", should_stop)
print("Status:", cap_test_state["status"])

print("\n=== SUCCESSFUL RUN UNCHANGED ===")
print("Status:", successful_negotiation["status"])
print("Agreed price:",
      successful_negotiation["agreed_price"])

assert should_stop is True
assert cap_test_state["status"] == "MAX_ROUNDS"
assert successful_negotiation["status"] == "DEAL"

print("\nPASS: Hard round cap enforced.")


=== BEFORE HARD-CAP CHECK ===
Round: 6
Status: ONGOING

=== AFTER HARD-CAP CHECK ===
Should stop: True
Status: MAX_ROUNDS

=== SUCCESSFUL RUN UNCHANGED ===
Status: DEAL
Agreed price: 405.0

PASS: Hard round cap enforced.


In [57]:

# Step 68: Reject an invalid ACCEPT decision

import copy

invalid_accept_state = {
    "round": 2,
    "status": "ONGOING",
    "history": [],
    "last_offer": 410.0,
    "last_offer_by": "seller",
    "agreed_price": None
}

# Buyer tries to accept $400 even though
# the Seller's outstanding offer is $410.
invalid_accept_response = {
    "decision": "ACCEPT",
    "price": 400.0,
    "message": "I accept your $400 offer."
}

print("=== INVALID ACCEPTANCE TEST ===")
print("Outstanding Seller offer: $410")
print("Buyer attempts to accept: $400")

try:
    apply_negotiation_decision(
        invalid_accept_state,
        "buyer",
        invalid_accept_response
    )

    print("FAIL: Invalid acceptance was allowed.")

except (ValueError, RuntimeError) as error:
    print("\nInvalid acceptance rejected:")
    print(type(error).__name__, ":", error)

print("\n=== STATE AFTER TEST ===")
print("Status:", invalid_accept_state["status"])
print("Agreed price:", invalid_accept_state["agreed_price"])
print("History entries:",
      len(invalid_accept_state["history"]))

assert invalid_accept_state["status"] == "ONGOING"
assert invalid_accept_state["agreed_price"] is None
assert len(invalid_accept_state["history"]) == 0

print("\nPASS: Invalid acceptance did not create a deal.")

print("\n=== REAL SUCCESSFUL RUN ===")
print("Status:", successful_negotiation["status"])
print("Agreed price: $",
      successful_negotiation["agreed_price"])


=== INVALID ACCEPTANCE TEST ===
Outstanding Seller offer: $410
Buyer attempts to accept: $400

Invalid acceptance rejected:
ValueError : Acceptance price must match outstanding offer

=== STATE AFTER TEST ===
Status: ONGOING
Agreed price: None
History entries: 0

PASS: Invalid acceptance did not create a deal.

=== REAL SUCCESSFUL RUN ===
Status: DEAL
Agreed price: $ 405.0


In [58]:

# Step 69: Test Buyer and Seller offer limits

def make_test_state():
    return {
        "round": 1,
        "status": "ONGOING",
        "history": [],
        "last_offer": None,
        "last_offer_by": None,
        "agreed_price": None
    }

test_cases = [
    (
        "Buyer exceeds $420 maximum",
        "buyer",
        500.0
    ),
    (
        "Seller goes below $400 minimum",
        "seller",
        350.0
    )
]

for description, agent_name, price in test_cases:
    test_state = make_test_state()

    response = {
        "decision": "OFFER",
        "price": price,
        "message": "Test offer."
    }

    print(f"\n=== {description} ===")

    try:
        apply_negotiation_decision(
            test_state,
            agent_name,
            response
        )
    except ValueError as error:
        print("Rejected:", error)
    else:
        raise AssertionError(
            f"Invalid {agent_name} offer was accepted!"
        )

    assert test_state["status"] == "ONGOING"
    assert test_state["last_offer"] is None
    assert len(test_state["history"]) == 0

    print("PASS: Invalid offer blocked.")

print("\n=== SUCCESSFUL RUN UNCHANGED ===")
print("Status:", successful_negotiation["status"])
print("Agreed price:",
      successful_negotiation["agreed_price"])

assert successful_negotiation["status"] == "DEAL"
assert successful_negotiation["agreed_price"] == 405.0

print("\nALL OFFER-LIMIT TESTS PASSED.")



=== Buyer exceeds $420 maximum ===
Rejected: INVALID OFFER: buyer cannot offer $500.0
PASS: Invalid offer blocked.

=== Seller goes below $400 minimum ===
Rejected: INVALID OFFER: seller cannot offer $350.0
PASS: Invalid offer blocked.

=== SUCCESSFUL RUN UNCHANGED ===
Status: DEAL
Agreed price: 405.0

ALL OFFER-LIMIT TESTS PASSED.


In [59]:

# Step 70: Audit web-search permissions

expected_agents = {
    "researcher",
    "buyer",
    "seller",
    "referee"
}

assert expected_agents.issubset(agents.keys())

print("=== AGENT TOOL PERMISSIONS ===")

web_enabled_agents = []

for agent_name, definition in agents.items():
    assigned_tools = list(definition.tools or [])

    has_web = "WebSearch" in assigned_tools

    print(
        f"{agent_name.upper():12} | "
        f"WebSearch: {has_web} | "
        f"Tools: {assigned_tools}"
    )

    if has_web:
        web_enabled_agents.append(agent_name)

print("\n=== PERMISSION CHECK ===")
print("Agents with WebSearch:",
      web_enabled_agents)

assert web_enabled_agents == ["researcher"], (
    "Exactly the Researcher must have WebSearch."
)

for name in ("buyer", "seller", "referee"):
    assert "WebSearch" not in (
        agents[name].tools or []
    )

print("\nPASS: Only Researcher has WebSearch.")
print("PASS: Buyer, Seller, and Referee have no web tool.")


=== AGENT TOOL PERMISSIONS ===
RESEARCHER   | WebSearch: True | Tools: ['WebSearch']
BUYER        | WebSearch: False | Tools: []
SELLER       | WebSearch: False | Tools: []
REFEREE      | WebSearch: False | Tools: []

=== PERMISSION CHECK ===
Agents with WebSearch: ['researcher']

PASS: Only Researcher has WebSearch.
PASS: Buyer, Seller, and Referee have no web tool.


In [60]:

# Step 71: Test the web-search permission hook

import inspect

print("=== WEB PERMISSION HOOK SIGNATURE ===")
print(inspect.signature(enforce_web_permissions))

print("\n=== WEB PERMISSION HOOK SOURCE ===")

try:
    print(inspect.getsource(enforce_web_permissions))
except (OSError, TypeError):
    print("Source unavailable in this Colab session.")

print("\n=== REGISTERED HOOKS ===")
print(web_permission_hooks)


=== WEB PERMISSION HOOK SIGNATURE ===
(input_data, tool_use_id, context)

=== WEB PERMISSION HOOK SOURCE ===
async def enforce_web_permissions(
    input_data,
    tool_use_id,
    context
):
    agent_type = input_data.get("agent_type", "")
    tool_name = input_data.get("tool_name", "")

    print(
        f"PERMISSION CHECK: "
        f"Agent={agent_type}, Tool={tool_name}"
    )

    if tool_name == "WebSearch":
        if agent_type == "researcher":
            print("WebSearch permitted for Researcher")
            return {
                "hookSpecificOutput": {
                    "hookEventName": "PreToolUse",
                    "permissionDecision": "allow"
                }
            }

        print("WebSearch blocked for non-Researcher")
        return {
            "hookSpecificOutput": {
                "hookEventName": "PreToolUse",
                "permissionDecision": "deny",
                "permissionDecisionReason":
                    "Only Researcher may use W

In [61]:

# Step 72: Verify WebSearch permission enforcement

test_agents = {
    "researcher": "allow",
    "buyer": "deny",
    "seller": "deny",
    "referee": "deny"
}

print("=== WEBSEARCH PERMISSION TEST ===")

for agent_name, expected_decision in test_agents.items():

    simulated_request = {
        "hook_event_name": "PreToolUse",
        "tool_name": "WebSearch",
        "tool_input": {"query": "used PS5 price"},
        "agent_type": agent_name
    }

    result = await enforce_web_permissions(
        simulated_request,
        None,
        {}
    )

    actual_decision = (
        result
        .get("hookSpecificOutput", {})
        .get("permissionDecision")
    )

    print(
        f"{agent_name.upper():12} | "
        f"Expected: {expected_decision:5} | "
        f"Actual: {actual_decision}"
    )

    assert actual_decision == expected_decision, (
        f"Permission test failed for {agent_name}"
    )

print("\nPASS: Researcher is allowed WebSearch.")
print("PASS: Buyer, Seller, and Referee are denied.")
print("PASS: Exclusive web-search policy enforced.")


=== WEBSEARCH PERMISSION TEST ===
PERMISSION CHECK: Agent=researcher, Tool=WebSearch
WebSearch permitted for Researcher
RESEARCHER   | Expected: allow | Actual: allow
PERMISSION CHECK: Agent=buyer, Tool=WebSearch
WebSearch blocked for non-Researcher
BUYER        | Expected: deny  | Actual: deny
PERMISSION CHECK: Agent=seller, Tool=WebSearch
WebSearch blocked for non-Researcher
SELLER       | Expected: deny  | Actual: deny
PERMISSION CHECK: Agent=referee, Tool=WebSearch
WebSearch blocked for non-Researcher
REFEREE      | Expected: deny  | Actual: deny

PASS: Researcher is allowed WebSearch.
PASS: Buyer, Seller, and Referee are denied.
PASS: Exclusive web-search policy enforced.


In [62]:

# Step 73: Summarize the real negotiation evidence

from collections import Counter

history = successful_negotiation["history"]

buyer_offers = [
    event["price"]
    for event in history
    if event["agent"] == "buyer"
    and event["decision"] == "OFFER"
]

seller_offers = [
    event["price"]
    for event in history
    if event["agent"] == "seller"
    and event["decision"] == "OFFER"
]

decisions = Counter(
    event["decision"] for event in history
)

print("=== REAL AGENT RUN ===")
print("Outcome:", successful_negotiation["status"])
print("Final price: $",
      successful_negotiation["agreed_price"])
print("Rounds:", successful_negotiation["round"])
print("Total decisions:", len(history))
print("SDK events:", len(successful_event_log))

print("\n=== OFFER PROGRESSION ===")
print("Buyer offers:", buyer_offers)
print("Seller offers:", seller_offers)

print("\n=== DECISION COUNTS ===")
for decision, count in sorted(decisions.items()):
    print(f"{decision}: {count}")

print("\n=== CONTROLLER SAFEGUARDS ===")
print("Maximum rounds:", MAX_ROUNDS)
print("Buyer maximum: $", BUYER_MAX)
print("Seller minimum: $", SELLER_MIN)

print("\n=== FINAL VALIDATION ===")
assert successful_negotiation["status"] == "DEAL"
assert successful_negotiation["agreed_price"] == 405.0
assert len(history) == 10
assert successful_negotiation["round"] <= MAX_ROUNDS

print("PASS: Successful run evidence is consistent.")


=== REAL AGENT RUN ===
Outcome: DEAL
Final price: $ 405.0
Rounds: 5
Total decisions: 10
SDK events: 86

=== OFFER PROGRESSION ===
Buyer offers: [340.0, 370.0, 395.0, 400.0, 405.0]
Seller offers: [440.0, 425.0, 415.0, 410.0]

=== DECISION COUNTS ===
ACCEPT: 1
OFFER: 9

=== CONTROLLER SAFEGUARDS ===
Maximum rounds: 6
Buyer maximum: $ 420
Seller minimum: $ 400

=== FINAL VALIDATION ===
PASS: Successful run evidence is consistent.


In [63]:

# Step 74: Reusable negotiation-state initializer

def create_negotiation_state():
    return {
        "round": 1,
        "status": "ONGOING",
        "history": [],
        "last_offer": None,
        "last_offer_by": None,
        "agreed_price": None
    }


# Test the initializer without changing our real run
fresh_state = create_negotiation_state()

print("=== FRESH NEGOTIATION STATE ===")
print(json.dumps(fresh_state, indent=2))

assert fresh_state["round"] == 1
assert fresh_state["status"] == "ONGOING"
assert fresh_state["history"] == []
assert fresh_state["last_offer"] is None
assert fresh_state["agreed_price"] is None

print("\nPASS: Fresh state initialized correctly.")

print("\n=== PRESERVED SUCCESSFUL RUN ===")
print("Status:", successful_negotiation["status"])
print("Agreed price:",
      successful_negotiation["agreed_price"])


=== FRESH NEGOTIATION STATE ===
{
  "round": 1,
  "status": "ONGOING",
  "history": [],
  "last_offer": null,
  "last_offer_by": null,
  "agreed_price": null
}

PASS: Fresh state initialized correctly.

=== PRESERVED SUCCESSFUL RUN ===
Status: DEAL
Agreed price: 405.0


In [64]:

# Step 75: Reusable Buyer–Seller negotiation round

async def run_negotiation_round(state):

    if check_negotiation_stop(state):
        print("Negotiation already stopped.")
        return state

    round_number = state["round"]

    print(f"\n=== ROUND {round_number} ===")

    # 1. Buyer makes an offer or responds to Seller
    buyer_prompt = f"""
    Ask the Buyer agent to negotiate in round {round_number}
    for a used standard disc PS5 with controller.

    COMPLETE HISTORY:
    {json.dumps(state["history"], indent=2)}

    OUTSTANDING OFFER:
    {state["last_offer"]}

    Buyer maximum: ${BUYER_MAX}
    Seller minimum: ${SELLER_MIN}
    Maximum rounds: {MAX_ROUNDS}

    Make your own decision.
    Do not invent a Seller response.

    Return exactly:
    Decision: OFFER, ACCEPT, or WALK_AWAY
    Price: $number
    Message: brief negotiation message
    """

    buyer_response = await call_negotiation_agent(
        "buyer", buyer_prompt
    )

    apply_negotiation_decision(
        state, "buyer", buyer_response
    )

    print("Buyer:", buyer_response)

    if check_negotiation_stop(state):
        return state

    # 2. Seller responds to Buyer's actual decision
    seller_prompt = f"""
    Ask the Seller agent to negotiate in round {round_number}
    for a used standard disc PS5 with controller.

    COMPLETE HISTORY:
    {json.dumps(state["history"], indent=2)}

    OUTSTANDING BUYER OFFER:
    ${state["last_offer"]:.2f}

    Buyer maximum: ${BUYER_MAX}
    Seller minimum: ${SELLER_MIN}
    Maximum rounds: {MAX_ROUNDS}

    Make your own decision.
    Do not invent a Buyer response.

    Return exactly:
    Decision: OFFER, ACCEPT, or WALK_AWAY
    Price: $number
    Message: brief negotiation message
    """

    seller_response = await call_negotiation_agent(
        "seller", seller_prompt
    )

    apply_negotiation_decision(
        state, "seller", seller_response
    )

    print("Seller:", seller_response)
    print("Status:", state["status"])

    return state


print("Reusable negotiation round function defined.")
print("No agents launched.")
print("Successful deal remains:",
      successful_negotiation["agreed_price"])


Reusable negotiation round function defined.
No agents launched.
Successful deal remains: 405.0


In [65]:

# Step 76: Automated negotiation loop

async def run_full_negotiation():
    state = create_negotiation_state()

    print("=== AUTOMATED NEGOTIATION STARTED ===")
    print("Maximum rounds:", MAX_ROUNDS)

    while not check_negotiation_stop(state):

        current_round = state["round"]

        await run_negotiation_round(state)

        # Stop immediately after a deal or walk-away
        if check_negotiation_stop(state):
            break

        # A full round completed without agreement
        state["round"] = current_round + 1

        # Enforce hard cap before launching more agents
        if check_negotiation_stop(state):
            break

    print("\n=== AUTOMATED NEGOTIATION FINISHED ===")
    print("Status:", state["status"])
    print("Rounds:", state["round"])
    print("Agreed price:", state["agreed_price"])
    print("Decisions:", len(state["history"]))

    return state


print("Automated negotiation loop defined.")
print("No API calls made.")
print("Preserved deal: $",
      successful_negotiation["agreed_price"])


Automated negotiation loop defined.
No API calls made.
Preserved deal: $ 405.0


In [66]:

# Step 77: Correct the six-round hard cap

async def run_full_negotiation():
    state = create_negotiation_state()

    print("=== AUTOMATED NEGOTIATION STARTED ===")
    print("Maximum rounds:", MAX_ROUNDS)

    for round_number in range(1, MAX_ROUNDS + 1):

        state["round"] = round_number

        # A deal or walk-away ends the negotiation
        if state["status"] != "ONGOING":
            break

        # Run one complete Buyer-Seller round.
        # Do not invoke the old round-cap check inside round 6.
        await run_negotiation_round(state)

        if state["status"] != "ONGOING":
            break

    # Only declare MAX_ROUNDS after the final round finishes
    if state["status"] == "ONGOING":
        state["status"] = "MAX_ROUNDS"

    print("\n=== AUTOMATED NEGOTIATION FINISHED ===")
    print("Status:", state["status"])
    print("Rounds:", state["round"])
    print("Agreed price:", state["agreed_price"])
    print("Decisions:", len(state["history"]))

    return state


print("Updated automated negotiation loop defined.")
print("No API calls made.")


Updated automated negotiation loop defined.
No API calls made.


In [67]:

# Step 78: Final-round-safe Buyer–Seller function

async def run_negotiation_round(state):

    if state["status"] != "ONGOING":
        print("Negotiation already stopped.")
        return state

    round_number = state["round"]

    assert 1 <= round_number <= MAX_ROUNDS

    print(f"\n=== ROUND {round_number} ===")

    buyer_prompt = f"""
    Ask the Buyer agent to negotiate round {round_number}
    for a used standard disc PS5 with controller.

    COMPLETE HISTORY:
    {json.dumps(state["history"], indent=2)}

    OUTSTANDING OFFER:
    {state["last_offer"]}

    Buyer maximum: ${BUYER_MAX}
    Seller minimum: ${SELLER_MIN}
    Maximum rounds: {MAX_ROUNDS}

    Make your own decision.
    Do not invent a Seller response.

    Return exactly:
    Decision: OFFER, ACCEPT, or WALK_AWAY
    Price: $number
    Message: brief negotiation message
    """

    buyer_response = await call_negotiation_agent(
        "buyer", buyer_prompt
    )

    apply_negotiation_decision(
        state, "buyer", buyer_response
    )

    print("Buyer:", buyer_response)

    # Stop on agreement or walk-away, not merely round number
    if state["status"] != "ONGOING":
        return state

    seller_prompt = f"""
    Ask the Seller agent to negotiate round {round_number}
    for a used standard disc PS5 with controller.

    COMPLETE HISTORY:
    {json.dumps(state["history"], indent=2)}

    OUTSTANDING BUYER OFFER:
    {state["last_offer"]}

    Buyer maximum: ${BUYER_MAX}
    Seller minimum: ${SELLER_MIN}
    Maximum rounds: {MAX_ROUNDS}

    Make your own decision.
    Do not invent a Buyer response.

    Return exactly:
    Decision: OFFER, ACCEPT, or WALK_AWAY
    Price: $number
    Message: brief negotiation message
    """

    seller_response = await call_negotiation_agent(
        "seller", seller_prompt
    )

    apply_negotiation_decision(
        state, "seller", seller_response
    )

    print("Seller:", seller_response)
    print("Status:", state["status"])

    return state


print("Final-round-safe function defined.")
print("No API calls made.")
print("Preserved successful deal: $",
      successful_negotiation["agreed_price"])


Final-round-safe function defined.
No API calls made.
Preserved successful deal: $ 405.0


In [68]:

# Step 79: Test the automated six-round boundary

real_round_function = run_negotiation_round
simulated_rounds = []

async def simulated_round(state):
    simulated_rounds.append(state["round"])
    print(f"Simulated round {state['round']} completed")
    return state

try:
    run_negotiation_round = simulated_round

    simulated_result = await run_full_negotiation()

finally:
    # Restore the actual SDK-powered function
    run_negotiation_round = real_round_function

print("\n=== HARD CAP TEST ===")
print("Rounds executed:", simulated_rounds)
print("Final status:", simulated_result["status"])

assert simulated_rounds == [1, 2, 3, 4, 5, 6]
assert simulated_result["status"] == "MAX_ROUNDS"
assert simulated_result["round"] == 6

print("\nPASS: All six permitted rounds executed.")
print("PASS: No seventh round launched.")
print("PASS: Hard cap enforced after round 6.")
print("PASS: Real SDK round function restored.")

print("\nPreserved successful deal: $",
      successful_negotiation["agreed_price"])


=== AUTOMATED NEGOTIATION STARTED ===
Maximum rounds: 6
Simulated round 1 completed
Simulated round 2 completed
Simulated round 3 completed
Simulated round 4 completed
Simulated round 5 completed
Simulated round 6 completed

=== AUTOMATED NEGOTIATION FINISHED ===
Status: MAX_ROUNDS
Rounds: 6
Agreed price: None
Decisions: 0

=== HARD CAP TEST ===
Rounds executed: [1, 2, 3, 4, 5, 6]
Final status: MAX_ROUNDS

PASS: All six permitted rounds executed.
PASS: No seventh round launched.
PASS: Hard cap enforced after round 6.
PASS: Real SDK round function restored.

Preserved successful deal: $ 405.0


In [69]:

# Step 80: Prepare independent evidence for the next SDK run

import copy
import inspect

print("=== ORIGINAL SUCCESSFUL RUN ===")
print("Status:", successful_negotiation["status"])
print("Deal: $", successful_negotiation["agreed_price"])
print("Saved SDK events:", len(successful_event_log))

# Verify the simulation has been removed
assert run_negotiation_round is real_round_function

# Verify required functions are available
assert inspect.iscoroutinefunction(run_negotiation_round)
assert inspect.iscoroutinefunction(call_negotiation_agent)
assert inspect.iscoroutinefunction(run_full_negotiation)

# Preserve the original log before starting a new run
original_event_count = len(successful_event_log)

# Create an independent event log for the upcoming run
negotiation_events = []

print("\n=== NEW RUN PREPARATION ===")
print("New event log length:", len(negotiation_events))
print("Original event log length:", len(successful_event_log))

assert len(negotiation_events) == 0
assert len(successful_event_log) == original_event_count

print("\nPASS: Real SDK functions are active.")
print("PASS: New event log is empty.")
print("PASS: Original successful evidence is preserved.")
print("No API calls made.")


=== ORIGINAL SUCCESSFUL RUN ===
Status: DEAL
Deal: $ 405.0
Saved SDK events: 86

=== NEW RUN PREPARATION ===
New event log length: 0
Original event log length: 86

PASS: Real SDK functions are active.
PASS: New event log is empty.
PASS: Original successful evidence is preserved.
No API calls made.


In [70]:

# Step 81: Live SDK negotiation preflight

print("=== LIVE RUN PREFLIGHT ===")

# Required agents
required_agents = {"researcher", "buyer", "seller", "referee"}
assert required_agents.issubset(agents.keys())
print("PASS: All four specialist agents are defined.")

# Exclusive web access
web_agents = [
    name for name, definition in agents.items()
    if "WebSearch" in (definition.tools or [])
]

assert web_agents == ["researcher"]
print("PASS: Only Researcher has WebSearch.")

# Negotiation limits
assert BUYER_MAX == 420
assert SELLER_MIN == 400
assert MAX_ROUNDS == 6
assert SELLER_MIN <= BUYER_MAX
print("PASS: Price limits and round cap are valid.")

# Controller functions
assert callable(create_negotiation_state)
assert callable(apply_negotiation_decision)
assert callable(parse_negotiation_response)
assert callable(validate_deal)
assert callable(log_sdk_event)
print("PASS: Controller and logging functions are available.")

# Clean evidence collection
assert negotiation_events == []
assert len(successful_event_log) == 86
print("PASS: New event log is clean; original log is preserved.")

# Check hook registration
assert "PreToolUse" in web_permission_hooks
assert any(
    matcher.matcher == "WebSearch"
    for matcher in web_permission_hooks["PreToolUse"]
)
print("PASS: WebSearch permission hook is registered.")

# Check fresh-state creation
test_state = create_negotiation_state()
assert test_state["status"] == "ONGOING"
assert test_state["round"] == 1
assert test_state["history"] == []
print("PASS: Negotiation starts from a fresh state.")

print("\nREADY: Live SDK negotiation preflight passed.")
print("No API calls made.")


=== LIVE RUN PREFLIGHT ===
PASS: All four specialist agents are defined.
PASS: Only Researcher has WebSearch.
PASS: Price limits and round cap are valid.
PASS: Controller and logging functions are available.
PASS: New event log is clean; original log is preserved.
PASS: WebSearch permission hook is registered.
PASS: Negotiation starts from a fresh state.

READY: Live SDK negotiation preflight passed.
No API calls made.


In [71]:

# Step 82: Execute the live automated SDK negotiation

print("=== LAUNCHING LIVE SDK NEGOTIATION ===")
print("Agents: Coordinator → Buyer / Seller")
print("Maximum rounds:", MAX_ROUNDS)
print("Buyer maximum: $", BUYER_MAX)
print("Seller minimum: $", SELLER_MIN)

automated_negotiation = None

try:
    automated_negotiation = await run_full_negotiation()

except Exception as error:
    print("\n=== LIVE RUN ERROR ===")
    print(type(error).__name__ + ":", error)
    print("SDK events captured:", len(negotiation_events))
    raise

print("\n=== LIVE RUN SUMMARY ===")
print("Status:", automated_negotiation["status"])
print("Agreed price:", automated_negotiation["agreed_price"])
print("Rounds:", automated_negotiation["round"])
print("Decisions:", len(automated_negotiation["history"]))
print("SDK events:", len(negotiation_events))

assert automated_negotiation["round"] <= MAX_ROUNDS

if automated_negotiation["status"] == "DEAL":
    assert validate_deal(
        automated_negotiation["agreed_price"]
    )
    print("PASS: Deal satisfies both price limits.")

elif automated_negotiation["status"] == "MAX_ROUNDS":
    print("PASS: Negotiation ended at the hard round cap.")

elif automated_negotiation["status"] == "WALK_AWAY":
    print("PASS: Negotiation ended by agent walk-away.")

else:
    raise AssertionError(
        "Unexpected negotiation status: "
        + str(automated_negotiation["status"])
    )

print("\nOriginal successful deal: $",
      successful_negotiation["agreed_price"])


=== LAUNCHING LIVE SDK NEGOTIATION ===
Agents: Coordinator → Buyer / Seller
Maximum rounds: 6
Buyer maximum: $ 420
Seller minimum: $ 400
=== AUTOMATED NEGOTIATION STARTED ===
Maximum rounds: 6

=== ROUND 1 ===
[tool_call] {'time': '2026-11-17T01:53:39.666543+00:00', 'type': 'tool_call', 'tool': 'Agent', 'input': {'subagent_type': 'buyer', 'description': 'Buyer negotiates round 1', 'prompt': 'Ask the Buyer agent to negotiate round 1 for a used standard disc PS5 with controller.\n\nCOMPLETE HISTORY:\n[]\n\nOUTSTANDING OFFER:\nNone\n\nBuyer maximum: $420\nSeller minimu
[user_or_tool_response] {'time': '2026-11-17T01:53:39.718905+00:00', 'type': 'user_or_tool_response', 'parent_id': None, 'content': '[ToolResultBlock(tool_use_id=\'toolu_014iwq6ChjJwXx3nkJFSFPQF\', content=[{\'type\': \'text\', \'text\': "Async agent launched successfully. (This tool result is internal metadata — never quote or paste any part of it, including the agentId 
[agent_text] {'time': '2026-11-17T01:53:40.732854+00

In [72]:

# Step 83: Preserve automated negotiation evidence

import copy

automated_successful_negotiation = copy.deepcopy(
    automated_negotiation
)

automated_successful_event_log = copy.deepcopy(
    negotiation_events
)

print("=== AUTOMATED RUN SAVED ===")
print("Status:", automated_successful_negotiation["status"])
print("Agreed price: $",
      automated_successful_negotiation["agreed_price"])
print("Rounds:", automated_successful_negotiation["round"])
print("Decisions:",
      len(automated_successful_negotiation["history"]))
print("SDK events:",
      len(automated_successful_event_log))

assert automated_successful_negotiation["status"] == "DEAL"
assert automated_successful_negotiation["agreed_price"] == 401.0
assert len(automated_successful_negotiation["history"]) == 12
assert len(automated_successful_event_log) == 74

print("\nPASS: Automated run evidence preserved.")
print("Original $405 deal remains preserved.")
print("No API calls made.")


=== AUTOMATED RUN SAVED ===
Status: DEAL
Agreed price: $ 401.0
Rounds: 6
Decisions: 12
SDK events: 74

PASS: Automated run evidence preserved.
Original $405 deal remains preserved.
No API calls made.


In [73]:

# Step 84: Independent Referee validation of the $401 deal

referee_prompt = f"""
Delegate this audit to the referee agent.

Audit the following completed used-PS5 negotiation.

RULES:
- Buyer maximum: ${BUYER_MAX}
- Seller minimum: ${SELLER_MIN}
- Maximum rounds: {MAX_ROUNDS}
- A valid ACCEPT must match the other party's outstanding offer.
- A DEAL must satisfy both price limits.
- Buyer and Seller decisions must alternate correctly.
- Do not invent missing decisions or change the history.

FINAL STATE:
{json.dumps(automated_successful_negotiation, indent=2)}

Check every decision and return:

Verdict: VALID or INVALID
Status: DEAL, MAX_ROUNDS, or WALK_AWAY
Reason: Explain your findings, including the final acceptance.
"""

referee_texts = []

print("=== LIVE REFEREE AUDIT ===")

async for message in query(
    prompt=referee_prompt,
    options=referee_options
):
    log_sdk_event(message)

    if isinstance(message, AssistantMessage):
        for block in message.content:
            if isinstance(block, TextBlock):
                referee_texts.append(block.text)

# Keep the last complete verdict, preferring the final response
valid_reports = [
    text for text in referee_texts
    if "Verdict:" in text and "Status:" in text
]

assert valid_reports, "Referee did not return a structured verdict."

automated_referee_report = valid_reports[-1]

print("\n=== REFEREE REPORT ===")
print(automated_referee_report)

print("\nPASS: Referee returned a structured audit.")


=== LIVE REFEREE AUDIT ===
[tool_call] {'time': '2026-11-17T01:56:51.882777+00:00', 'type': 'tool_call', 'tool': 'Agent', 'input': {'subagent_type': 'referee', 'description': 'Audit used-PS5 negotiation', 'prompt': 'Audit the following completed used-PS5 negotiation. Check every decision and return:\n\nVerdict: VALID or INVALID\nStatus: DEAL, MAX_ROUNDS, or WALK_AWAY\nReason: Explain yo
[user_or_tool_response] {'time': '2026-11-17T01:56:51.928189+00:00', 'type': 'user_or_tool_response', 'parent_id': None, 'content': '[ToolResultBlock(tool_use_id=\'toolu_01QtU2jCuVKhGd8WsMxSm1VP\', content=[{\'type\': \'text\', \'text\': "Async agent launched successfully. (This tool result is internal metadata — never quote or paste any part of it, including the agentId 
[agent_text] {'time': '2026-11-17T01:56:53.437634+00:00', 'type': 'agent_text', 'text': "The Referee is auditing the negotiation now, and I'll report its verdict when it comes back. I haven't audited the negotiation myself.\n\nI sent t

In [74]:

# Step 85: Complete and verify automated-run evidence

import copy
import re

automated_evidence = {
    "run_type": "real_anthropic_agent_sdk",
    "negotiation": copy.deepcopy(
        automated_successful_negotiation
    ),
    "sdk_events": copy.deepcopy(
        negotiation_events
    ),
    "referee_report": automated_referee_report,
    "limits": {
        "buyer_max": BUYER_MAX,
        "seller_min": SELLER_MIN,
        "max_rounds": MAX_ROUNDS
    }
}

report = automated_evidence["referee_report"]

assert re.search(
    r"Verdict:\s*\*{0,2}VALID\b",
    report,
    re.IGNORECASE
)

assert re.search(
    r"Status:\s*\*{0,2}DEAL\b",
    report,
    re.IGNORECASE
)

assert automated_evidence["negotiation"]["agreed_price"] == 401.0
assert len(automated_evidence["negotiation"]["history"]) == 12

print("=== COMPLETE AUTOMATED EVIDENCE ===")
print("Run type:", automated_evidence["run_type"])
print("Status:", automated_evidence["negotiation"]["status"])
print("Agreed price: $",
      automated_evidence["negotiation"]["agreed_price"])
print("Decisions:",
      len(automated_evidence["negotiation"]["history"]))
print("SDK events (including Referee):",
      len(automated_evidence["sdk_events"]))
print("Referee verdict: VALID")

print("\nPASS: Negotiation and Referee evidence preserved.")
print("No API calls made.")


=== COMPLETE AUTOMATED EVIDENCE ===
Run type: real_anthropic_agent_sdk
Status: DEAL
Agreed price: $ 401.0
Decisions: 12
SDK events (including Referee): 81
Referee verdict: VALID

PASS: Negotiation and Referee evidence preserved.
No API calls made.


In [75]:

# Step 86: Save both real SDK runs to Google Drive

from google.colab import drive
from pathlib import Path
import json
import copy

drive.mount("/content/drive")

evidence_folder = Path(
    "/content/drive/MyDrive/PS5_Agent_Assignment/evidence"
)
evidence_folder.mkdir(parents=True, exist_ok=True)

original_evidence = {
    "run_type": "real_anthropic_agent_sdk",
    "negotiation": copy.deepcopy(successful_negotiation),
    "sdk_events": copy.deepcopy(successful_event_log),
    "referee_report": successful_referee_report
}

files_to_save = {
    "original_405_run.json": original_evidence,
    "automated_401_run.json": automated_evidence
}

for filename, data in files_to_save.items():
    file_path = evidence_folder / filename

    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(
            data,
            f,
            indent=2,
            ensure_ascii=False,
            default=str
        )

    print(
        f"Saved: {filename} "
        f"({file_path.stat().st_size:,} bytes)"
    )

print("\n=== PERSISTENT EVIDENCE ===")
print("Folder:", evidence_folder)
print("Files saved:", len(files_to_save))

assert all(
    (evidence_folder / filename).exists()
    for filename in files_to_save
)

print("\nPASS: Both SDK runs saved to Google Drive.")
print("PASS: Evidence survives Colab runtime restarts.")


Mounted at /content/drive
Saved: original_405_run.json (78,232 bytes)
Saved: automated_401_run.json (81,061 bytes)

=== PERSISTENT EVIDENCE ===
Folder: /content/drive/MyDrive/PS5_Agent_Assignment/evidence
Files saved: 2

PASS: Both SDK runs saved to Google Drive.
PASS: Evidence survives Colab runtime restarts.


In [76]:

# Step 87: Researcher integration preflight

print("=== RESEARCHER INTEGRATION CHECK ===")

assert "researcher" in agents
assert "WebSearch" in (agents["researcher"].tools or [])
print("PASS: Researcher has WebSearch.")

assert isinstance(research_report, str)
assert research_report.strip()
print("PASS: Research report is available.")
print("Research report length:", len(research_report), "characters")

assert isinstance(market_context, str)
assert market_context.strip()
print("PASS: Market context is available.")

# Verify Buyer and Seller receive research context
for agent_name in ("buyer", "seller"):
    agent_prompt = agents[agent_name].prompt

    assert isinstance(agent_prompt, str)
    assert market_context.strip() in agent_prompt, (
        f"{agent_name} prompt does not contain market context."
    )

    print(
        f"PASS: {agent_name.capitalize()} "
        "has the market context in its instructions."
    )

print("\n=== RESEARCH PREVIEW ===")
print(research_report[:600])

print("\nREADY: Researcher context is connected to both negotiators.")
print("No API calls made.")


=== RESEARCHER INTEGRATION CHECK ===
PASS: Researcher has WebSearch.
PASS: Research report is available.
Research report length: 1826 characters
PASS: Market context is available.
PASS: Buyer has the market context in its instructions.
PASS: Seller has the market context in its instructions.

=== RESEARCH PREVIEW ===
The Researcher found 2026 sold prices for used US standard disc PS5s, but it could only read search-result snippets. It did not open any of the listing or sold-history pages, and it found no complete dataset. Treat the results as low confidence.

**Sold prices, working used standard disc PS5 (CFI-1215A, with controller):**
- $455.00, sold September 5, 2026 (eBay, 69 bids). Source: https://www.ebay.com/itm/168648392090
- $479.99, sold November 6 (year not shown, presumably 2026). Source: https://www.ebay.com/itm/168648392090
- $395.00, sold September 12, 2026, with two controllers and accessori

READY: Researcher context is connected to both negotiators.
No API calls made.


In [77]:

# Step 88: Define the integrated workflow entry point
# Defining this function makes no API calls.

async def run_five_agent_workflow():

    print("=== FIVE-AGENT WORKFLOW ===")

    # Researcher output is already embedded in the
    # Buyer and Seller agent definitions.
    assert research_report.strip()
    assert market_context.strip()

    print("Researcher: existing market report available")
    print("Coordinator: delegates through Anthropic SDK")

    # Run the actual Buyer-Seller negotiation
    state = await run_full_negotiation()

    # Ask the Referee to audit the completed state
    audit_prompt = f"""
    Delegate this audit to the referee agent.

    Validate the negotiation using these rules:
    Buyer maximum: ${BUYER_MAX}
    Seller minimum: ${SELLER_MIN}
    Maximum rounds: {MAX_ROUNDS}
    ACCEPT must match the other party's outstanding offer.
    Decisions must alternate Buyer, Seller.

    NEGOTIATION STATE:
    {json.dumps(state, indent=2)}

    Return:
    Verdict: VALID or INVALID
    Status: DEAL, MAX_ROUNDS, or WALK_AWAY
    Reason: Explain your findings.
    """

    audit_texts = []

    async for message in query(
        prompt=audit_prompt,
        options=referee_options
    ):
        log_sdk_event(message)

        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    audit_texts.append(block.text)

    reports = [
        text for text in audit_texts
        if "Verdict:" in text and "Status:" in text
    ]

    if not reports:
        raise RuntimeError(
            "Referee did not return a structured audit."
        )

    referee_report = reports[-1]

    print("\n=== FINAL REFEREE REPORT ===")
    print(referee_report)

    return {
        "negotiation": state,
        "research_report": research_report,
        "referee_report": referee_report
    }


print("Integrated workflow function defined.")
print("No API calls made.")
print("Existing $401 and $405 evidence remains preserved.")


Integrated workflow function defined.
No API calls made.
Existing $401 and $405 evidence remains preserved.


In [78]:

# Step 89: Inspect the existing Researcher SDK setup

print("=== RESEARCHER CONFIGURATION ===")

print("Researcher tools:",
      agents["researcher"].tools)

print("\n=== AVAILABLE RESEARCH OPTIONS ===")

for variable_name in (
    "research_options",
    "researcher_options",
    "web_permission_hooks"
):
    if variable_name in globals():
        value = globals()[variable_name]
        print(f"{variable_name}: {type(value).__name__}")

        if isinstance(value, ClaudeAgentOptions):
            print("  Model:", value.model)
            print("  Allowed tools:", value.allowed_tools)
            print("  Agent definitions:",
                  list((value.agents or {}).keys()))
            print("  Hooks configured:", bool(value.hooks))
    else:
        print(f"{variable_name}: not defined")

print("\n=== EXISTING RESEARCH ===")
print("Report available:", bool(research_report.strip()))
print("Market context available:", bool(market_context.strip()))

print("\nInspection complete. No API calls made.")


=== RESEARCHER CONFIGURATION ===
Researcher tools: ['WebSearch']

=== AVAILABLE RESEARCH OPTIONS ===
research_options: not defined
researcher_options: not defined
web_permission_hooks: dict

=== EXISTING RESEARCH ===
Report available: True
Market context available: True

Inspection complete. No API calls made.


In [79]:

# Step 90: Configure the Researcher for live SDK searches

research_options = ClaudeAgentOptions(
    model="haiku",
    allowed_tools=["Task"],
    agents={
        "researcher": agents["researcher"]
    },
    hooks=web_permission_hooks,
    max_turns=8,
    system_prompt="""
    You are the Coordinator for marketplace research.

    Delegate research tasks to the researcher agent.
    Do not conduct web searches yourself.

    The Researcher is the only agent authorized to use
    WebSearch.

    Wait for the Researcher's findings before reporting
    the result.

    Preserve source URLs and clearly distinguish verified
    evidence from uncertain search-result snippets.
    """
)

print("=== RESEARCHER SDK CONFIGURATION ===")
print("Model:", research_options.model)
print("Coordinator tools:", research_options.allowed_tools)
print("Delegated agents:",
      list(research_options.agents.keys()))
print("Researcher tools:",
      research_options.agents["researcher"].tools)
print("Permission hooks enabled:",
      bool(research_options.hooks))

assert research_options.allowed_tools == ["Task"]
assert list(research_options.agents.keys()) == ["researcher"]
assert research_options.agents["researcher"].tools == ["WebSearch"]
assert "PreToolUse" in research_options.hooks

print("\nPASS: Researcher delegation configured.")
print("PASS: Exclusive WebSearch permission hook attached.")
print("No API calls made.")


=== RESEARCHER SDK CONFIGURATION ===
Model: haiku
Coordinator tools: ['Task']
Delegated agents: ['researcher']
Researcher tools: ['WebSearch']
Permission hooks enabled: True

PASS: Researcher delegation configured.
PASS: Exclusive WebSearch permission hook attached.
No API calls made.


In [80]:

# Step 91: Reusable live Researcher function

async def run_live_research():

    research_prompt = """
    Delegate this marketplace research task to the researcher agent.

    Research recent SOLD prices for a used standard disc-edition
    PlayStation 5 (PS5) with one working controller in the US.

    Requirements:
    - Use WebSearch to find relevant sold listings.
    - Prefer completed or sold listings over asking prices.
    - Include prices, dates, condition, and source URLs.
    - Distinguish actual verified listings from search snippets.
    - Do not invent sold prices or listing details.
    - State limitations and confidence clearly.
    - Provide a practical estimated negotiation range.

    Wait for the Researcher's final report.
    """

    research_texts = []

    print("=== LIVE RESEARCHER STARTED ===")

    async for message in query(
        prompt=research_prompt,
        options=research_options
    ):
        log_sdk_event(message)

        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    research_texts.append(block.text)

    if not research_texts:
        raise RuntimeError(
            "Researcher returned no text."
        )

    # Prefer the final substantive report, not an interim
    # Coordinator message saying the agent is still running.
    reports = [
        text for text in research_texts
        if len(text.strip()) >= 300
        and (
            "http" in text.lower()
            or "source" in text.lower()
        )
    ]

    if not reports:
        raise RuntimeError(
            "No substantive research report found. "
            "Inspect the streamed SDK events."
        )

    final_report = reports[-1]

    print("\n=== RESEARCHER REPORT ===")
    print(final_report)

    return final_report


print("Reusable live Researcher function defined.")
print("WebSearch hook attached through research_options.")
print("No API calls made.")


Reusable live Researcher function defined.
WebSearch hook attached through research_options.
No API calls made.


In [81]:

# Step 92: Update negotiators with fresh research

from dataclasses import replace

# Save the original instructions as reusable templates.
# This cell should be run only once.
original_buyer_prompt = agents["buyer"].prompt
original_seller_prompt = agents["seller"].prompt
original_market_context = market_context


def update_negotiator_research(new_report):
    global market_context

    assert isinstance(new_report, str)
    assert new_report.strip()

    new_context = (
        "MARKET RESEARCH (Researcher agent):\n"
        + new_report.strip()
    )

    for name, original_prompt in (
        ("buyer", original_buyer_prompt),
        ("seller", original_seller_prompt)
    ):
        assert original_market_context in original_prompt, (
            f"Original research context missing from {name}."
        )

        updated_prompt = original_prompt.replace(
            original_market_context,
            new_context
        )

        agents[name] = replace(
            agents[name],
            prompt=updated_prompt
        )

    market_context = new_context

    print("Buyer and Seller research context updated.")
    print("New report length:", len(new_report))


print("Research-context updater defined.")
print("Existing agent instructions unchanged.")
print("No API calls made.")


Research-context updater defined.
Existing agent instructions unchanged.
No API calls made.


In [82]:

# Step 93: Test research updates without changing real agents

from dataclasses import replace

test_report = (
    "TEST RESEARCH ONLY: Used PS5 market estimate $390-$450. "
    "This is synthetic test data, not a real market finding."
)

test_context = (
    "MARKET RESEARCH (Researcher agent):\n"
    + test_report
)

print("=== RESEARCH UPDATE UNIT TEST ===")

for name, original_prompt in (
    ("buyer", original_buyer_prompt),
    ("seller", original_seller_prompt)
):
    assert original_market_context in original_prompt

    updated_prompt = original_prompt.replace(
        original_market_context,
        test_context
    )

    temporary_agent = replace(
        agents[name],
        prompt=updated_prompt
    )

    assert test_context in temporary_agent.prompt
    assert original_market_context not in temporary_agent.prompt
    assert temporary_agent.tools == agents[name].tools

    print(f"PASS: {name.capitalize()} research replaced.")
    print(f"PASS: {name.capitalize()} tool permissions preserved.")

# Confirm the actual agents were not modified
assert agents["buyer"].prompt == original_buyer_prompt
assert agents["seller"].prompt == original_seller_prompt

print("\nPASS: Real Buyer and Seller remain unchanged.")
print("PASS: Synthetic research was not used in negotiation.")
print("No API calls made.")


=== RESEARCH UPDATE UNIT TEST ===
PASS: Buyer research replaced.
PASS: Buyer tool permissions preserved.
PASS: Seller research replaced.
PASS: Seller tool permissions preserved.

PASS: Real Buyer and Seller remain unchanged.
PASS: Synthetic research was not used in negotiation.
No API calls made.


In [83]:

# Step 94: Integrate Researcher with the five-agent workflow

async def run_five_agent_workflow(fresh_research=False):
    global research_report

    print("=== FIVE-AGENT WORKFLOW ===")

    if fresh_research:
        print("\nPhase 1: Fresh Researcher search")

        new_report = await run_live_research()

        # Update the negotiators only after research succeeds
        update_negotiator_research(new_report)
        research_report = new_report

    else:
        print("\nPhase 1: Reusing existing market research")
        assert research_report.strip()
        assert market_context.strip()

    print("\nPhase 2: Buyer/Seller negotiation")
    final_state = await run_full_negotiation()

    print("\nPhase 3: Independent Referee review")

    referee_prompt = (
        "You are the independent Referee.\n"
        "Audit the following negotiation record.\n"
        "Check price limits, alternating turns, acceptance "
        "of exact outstanding offers, round cap, and final status.\n"
        "Report VALID or INVALID with reasons.\n\n"
        + json.dumps(final_state, indent=2)
    )

    referee_texts = []

    async for message in query(
        prompt=referee_prompt,
        options=referee_options
    ):
        log_sdk_event(message)

        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    referee_texts.append(block.text)

    if not referee_texts:
        raise RuntimeError("Referee returned no text.")

    referee_report = referee_texts[-1]

    print("\n=== REFEREE REPORT ===")
    print(referee_report)

    return {
        "research_report": research_report,
        "negotiation": final_state,
        "referee_report": referee_report
    }


print("Five-agent workflow updated.")
print("Fresh research is optional.")
print("No API calls made.")


Five-agent workflow updated.
Fresh research is optional.
No API calls made.


In [84]:

# Step 95: Full workflow readiness check (no API calls)

import inspect
import json

print("=== FIVE-AGENT WORKFLOW READINESS ===")

required_functions = [
    "run_live_research",
    "update_negotiator_research",
    "run_full_negotiation",
    "run_five_agent_workflow",
    "log_sdk_event",
]

for name in required_functions:
    assert name in globals() and callable(globals()[name]), (
        f"Missing function: {name}"
    )
    print(f"PASS: {name}")

assert inspect.iscoroutinefunction(run_five_agent_workflow)
assert inspect.iscoroutinefunction(run_live_research)

assert all(
    name in agents
    for name in ("researcher", "buyer", "seller", "referee")
)
print("PASS: All four delegated agents available")

assert research_options.agents["researcher"].tools == ["WebSearch"]
assert "PreToolUse" in research_options.hooks
print("PASS: Researcher WebSearch permissions configured")

assert research_report.strip()
assert market_context.strip()
assert market_context in agents["buyer"].prompt
assert market_context in agents["seller"].prompt
print("PASS: Market research available to both negotiators")

assert "referee_options" in globals()
assert isinstance(referee_options, ClaudeAgentOptions)
print("PASS: Referee SDK options available")

assert MAX_ROUNDS == 6
assert BUYER_MAX == 420
assert SELLER_MIN == 400
print("PASS: Negotiation limits configured")

assert "successful_negotiation" in globals()
assert "automated_successful_negotiation" in globals()
print("PASS: Both successful SDK runs preserved")

print("\nREADY: All required workflow dependencies found.")
print("No API calls made.")


=== FIVE-AGENT WORKFLOW READINESS ===
PASS: run_live_research
PASS: update_negotiator_research
PASS: run_full_negotiation
PASS: run_five_agent_workflow
PASS: log_sdk_event
PASS: All four delegated agents available
PASS: Researcher WebSearch permissions configured
PASS: Market research available to both negotiators
PASS: Referee SDK options available
PASS: Negotiation limits configured
PASS: Both successful SDK runs preserved

READY: All required workflow dependencies found.
No API calls made.


In [85]:

# Step 96: Audit saved negotiation evidence

import os
import json

evidence_dir = (
    "/content/drive/MyDrive/"
    "PS5_Agent_Assignment/evidence"
)

expected_files = {
    "original_405_run.json": 405,
    "automated_401_run.json": 401,
}

print("=== SAVED EVIDENCE AUDIT ===")

for filename, expected_price in expected_files.items():
    path = os.path.join(evidence_dir, filename)

    assert os.path.isfile(path), f"Missing: {filename}"

    with open(path, "r", encoding="utf-8") as f:
        evidence = json.load(f)

    state = evidence["negotiation"]

    assert state["status"] == "DEAL", (
        f"{filename}: expected DEAL status"
    )
    assert float(state["agreed_price"]) == expected_price, (
        f"{filename}: unexpected agreed price"
    )

    assert 1 <= state["round"] <= MAX_ROUNDS

    print(f"\nPASS: {filename}")
    print("  Status:", state["status"])
    print("  Agreed price: $", state["agreed_price"], sep="")
    print("  Final round:", state["round"])
    print("  File size:", os.path.getsize(path), "bytes")
    print("  Referee report present:",
          bool(evidence.get("referee_report")))

print("\nPASS: Both saved evidence files verified.")
print("No API calls made. No files modified.")


=== SAVED EVIDENCE AUDIT ===

PASS: original_405_run.json
  Status: DEAL
  Agreed price: $405.0
  Final round: 5
  File size: 78232 bytes
  Referee report present: True

PASS: automated_401_run.json
  Status: DEAL
  Agreed price: $401.0
  Final round: 6
  File size: 81061 bytes
  Referee report present: True

PASS: Both saved evidence files verified.
No API calls made. No files modified.


In [86]:

# Step 97: Submission-readiness audit

print("=== SUBMISSION READINESS AUDIT ===")

checks = {
    "Five agent roles": (
        all(x in agents for x in
            ["researcher", "buyer", "seller", "referee"])
    ),
    "Researcher has WebSearch": (
        agents["researcher"].tools == ["WebSearch"]
    ),
    "Buyer has no tools": (
        agents["buyer"].tools == []
    ),
    "Seller has no tools": (
        agents["seller"].tools == []
    ),
    "Referee has no tools": (
        agents["referee"].tools == []
    ),
    "Research permission hook": (
        "PreToolUse" in web_permission_hooks
    ),
    "Negotiation decision parser": (
        callable(parse_negotiation_response)
    ),
    "Negotiation rule enforcement": (
        callable(apply_negotiation_decision)
    ),
    "Hard round cap": (
        MAX_ROUNDS == 6
    ),
    "Automated negotiation loop": (
        callable(run_full_negotiation)
    ),
    "Research integration": (
        callable(update_negotiator_research)
    ),
    "Complete workflow": (
        callable(run_five_agent_workflow)
    ),
    "Original research template": (
        original_market_context in original_buyer_prompt
        and original_market_context in original_seller_prompt
    ),
    "Successful SDK evidence": (
        successful_negotiation["status"] == "DEAL"
        and automated_successful_negotiation["status"] == "DEAL"
    ),
}

for label, passed in checks.items():
    print(
        ("PASS" if passed else "FAIL")
        + ": " + label
    )

passed_count = sum(checks.values())

print(f"\nResult: {passed_count}/{len(checks)} checks passed.")

if passed_count == len(checks):
    print("READY for notebook organization.")
else:
    print("Resolve failed checks before final packaging.")

print("No API calls made.")


=== SUBMISSION READINESS AUDIT ===
PASS: Five agent roles
PASS: Researcher has WebSearch
PASS: Buyer has no tools
PASS: Seller has no tools
PASS: Referee has no tools
PASS: Research permission hook
PASS: Negotiation decision parser
PASS: Negotiation rule enforcement
PASS: Hard round cap
PASS: Automated negotiation loop
PASS: Research integration
PASS: Complete workflow
PASS: Original research template
PASS: Successful SDK evidence

Result: 14/14 checks passed.
READY for notebook organization.
No API calls made.


In [87]:

# Step 98: Inspect current Colab notebook structure

import os
import glob

print("=== NOTEBOOK STRUCTURE INSPECTION ===")

notebooks = glob.glob(
    "/content/drive/MyDrive/**/*.ipynb",
    recursive=True
)

print("Notebooks found in Google Drive:", len(notebooks))

for path in notebooks[:15]:
    print("-", os.path.relpath(
        path,
        "/content/drive/MyDrive"
    ))

print("\n=== CURRENT RUNTIME ===")
print("Agent definitions:", list(agents.keys()))
print("Research report available:", bool(research_report.strip()))
print("Saved SDK evidence files: 2 verified")

print("\nNo API calls made.")
print("No notebook files modified.")


=== NOTEBOOK STRUCTURE INSPECTION ===
Notebooks found in Google Drive: 8
- Colab Notebooks/Untitled0.ipynb
- Colab Notebooks/Untitled1.ipynb
- Colab Notebooks/Untitled2.ipynb
- Colab Notebooks/Untitled3.ipynb
- Colab Notebooks/malkakunage_shashika_assignment2.ipynb
- Colab Notebooks/Business_Concepts_Fine_Tuning.ipynb
- Colab Notebooks/Business_Concepts_Tutor_LoRA.ipynb
- Colab Notebooks/AI_Agents_Assignment.ipynb

=== CURRENT RUNTIME ===
Agent definitions: ['researcher', 'buyer', 'seller', 'referee']
Research report available: True
Saved SDK evidence files: 2 verified

No API calls made.
No notebook files modified.


In [88]:

# Step 99: Inspect assignment notebook cells (read-only)

import json
import os

notebook_path = (
    "/content/drive/MyDrive/Colab Notebooks/"
    "AI_Agents_Assignment.ipynb"
)

assert os.path.isfile(notebook_path), "Notebook not found."

with open(notebook_path, "r", encoding="utf-8") as f:
    notebook_data = json.load(f)

cells = notebook_data.get("cells", [])

code_count = sum(c["cell_type"] == "code" for c in cells)
markdown_count = sum(c["cell_type"] == "markdown" for c in cells)

print("=== ASSIGNMENT NOTEBOOK AUDIT ===")
print("Total cells:", len(cells))
print("Code cells:", code_count)
print("Markdown cells:", markdown_count)

print("\n=== CELL OUTLINE ===")

for index, cell in enumerate(cells):
    source = "".join(cell.get("source", []))
    nonempty_lines = [
        line.strip()
        for line in source.splitlines()
        if line.strip()
    ]

    preview = nonempty_lines[0] if nonempty_lines else "(empty)"
    preview = preview[:100]

    cell_type = "MD" if cell["cell_type"] == "markdown" else "CODE"

    print(f"{index:03d} [{cell_type}] {preview}")

print("\nInspection complete.")
print("No files modified. No API calls made.")


=== ASSIGNMENT NOTEBOOK AUDIT ===
Total cells: 90
Code cells: 87
Markdown cells: 3

=== CELL OUTLINE ===
000 [CODE] !pip install -q anthropic
001 [CODE] from google.colab import userdata
002 [CODE] import anthropic
003 [CODE] # Step 8: Define Buyer and Seller agents
004 [CODE] # Step 9: First negotiation test (Version 1)
005 [MD] ## Tuning Log — Version 1
006 [CODE] # Step 12: Improved Buyer and Seller — Version 2
007 [CODE] # Step 13: Version 2 negotiation loop
008 [MD] ## Tuning Log — Version 2
009 [CODE] # Step 16: Version 3 — Stronger Seller
010 [CODE] # Step 17: Version 3 negotiation test
011 [MD] ## Tuning Log — Version 3
012 [CODE] # Step 20: Install Anthropic Agent SDK
013 [CODE] # Step 21: Verify Claude Agent SDK
014 [CODE] # Step 22: Test Claude Agent SDK authentication
015 [CODE] # Step 23: Check bundled Claude executable
016 [CODE] # Step 24: Test Claude Agent SDK
017 [CODE] # Step 25: Define our five-agent architecture
018 [CODE] # Step 26: Test Coordinator -> Researcher d

In [89]:

# Step 100: Find function definitions and duplicates

import ast
from collections import defaultdict

definitions = defaultdict(list)

for index, cell in enumerate(cells):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))

    try:
        tree = ast.parse(source)
    except SyntaxError:
        continue

    for node in tree.body:
        if isinstance(
            node, (ast.FunctionDef, ast.AsyncFunctionDef)
        ):
            definitions[node.name].append(index)

print("=== FUNCTION DEFINITION AUDIT ===")

for name, locations in sorted(definitions.items()):
    if len(locations) > 1:
        print(
            f"DUPLICATE: {name} "
            f"-> cells {locations} "
            f"(latest: {locations[-1]})"
        )

print("\n=== IMPORTANT FINAL FUNCTIONS ===")

important = [
    "enforce_web_permissions",
    "log_sdk_event",
    "parse_negotiation_response",
    "apply_negotiation_decision",
    "check_negotiation_stop",
    "call_negotiation_agent",
    "create_negotiation_state",
    "run_negotiation_round",
    "run_full_negotiation",
    "run_live_research",
    "update_negotiator_research",
    "run_five_agent_workflow",
]

for name in important:
    locations = definitions.get(name, [])
    print(f"{name}: {locations or 'NOT FOUND'}")

print("\nRead-only inspection complete.")
print("No API calls made. No files modified.")


=== FUNCTION DEFINITION AUDIT ===
DUPLICATE: apply_negotiation_decision -> cells [36, 38] (latest: 38)
DUPLICATE: call_negotiation_agent -> cells [39, 48] (latest: 48)
DUPLICATE: research_permissions -> cells [19, 21] (latest: 21)
DUPLICATE: run_five_agent_workflow -> cells [79, 85] (latest: 85)
DUPLICATE: run_full_negotiation -> cells [67, 68] (latest: 68)
DUPLICATE: run_negotiation_round -> cells [66, 69] (latest: 69)

=== IMPORTANT FINAL FUNCTIONS ===
enforce_web_permissions: [25]
log_sdk_event: [33]
parse_negotiation_response: [35]
apply_negotiation_decision: [36, 38]
check_negotiation_stop: [34]
call_negotiation_agent: [39, 48]
create_negotiation_state: [65]
run_negotiation_round: [66, 69]
run_full_negotiation: [67, 68]
run_live_research: [82]
update_negotiator_research: [83]
run_five_agent_workflow: [79, 85]

Read-only inspection complete.
No API calls made. No files modified.


In [90]:

# Step 101: Inspect final controller dependencies

inspect_cells = [36, 38, 48, 68, 69]

print("=== FINAL CONTROLLER SOURCE AUDIT ===")

for index in inspect_cells:
    cell = cells[index]
    source = "".join(cell.get("source", []))

    print("\n" + "=" * 60)
    print(f"CELL {index}")
    print("=" * 60)
    print(source)

print("\nRead-only inspection complete.")
print("No API calls made. No files modified.")


=== FINAL CONTROLLER SOURCE AUDIT ===

CELL 36

# Step 45: Apply negotiation decisions safely

def apply_negotiation_decision(state, agent, response):
    if agent not in ("buyer", "seller"):
        raise ValueError("Unknown negotiation agent")

    if state["status"] != "ONGOING":
        raise ValueError("Negotiation has already ended")

    decision = response["decision"]
    price = response["price"]

    event = {
        "round": state["round"],
        "agent": agent,
        "decision": decision,
        "price": price,
        "message": response["message"]
    }

    if decision == "WALK_AWAY":
        state["status"] = "WALK_AWAY"

    elif decision == "OFFER":
        if price is None or price <= 0:
            raise ValueError("Invalid offer price")

        state["last_offer"] = price
        state["last_offer_by"] = agent

    elif decision == "ACCEPT":
        outstanding = state["last_offer"]

        if outstanding is None:
            raise ValueError("No outstandin

In [91]:

# Step 102: Inspect supporting negotiation definitions

inspect_cells = [34, 35, 37, 43, 65]

print("=== NEGOTIATION SUPPORT FUNCTIONS ===")

for index in inspect_cells:
    source = "".join(cells[index].get("source", []))

    print("\n" + "=" * 60)
    print(f"CELL {index}")
    print("=" * 60)
    print(source)

print("\nInspection complete.")
print("No API calls made. No files modified.")


=== NEGOTIATION SUPPORT FUNCTIONS ===

CELL 34

# Step 43: Negotiation state and stopping rules

MAX_ROUNDS = 6
BUYER_MAX = 420
SELLER_MIN = 400

negotiation_state = {
    "round": 0,
    "status": "ONGOING",
    "history": [],
    "last_offer": None,
    "last_offer_by": None,
    "agreed_price": None
}

def check_negotiation_stop(state):
    """
    Return True if negotiation should stop.
    """

    if state["status"] in ["DEAL", "WALK_AWAY"]:
        return True

    if state["round"] >= MAX_ROUNDS:
        state["status"] = "MAX_ROUNDS"
        return True

    return False

def validate_deal(price):
    """
    Check whether a proposed deal respects
    both parties' hard price limits.
    """
    return (
        price is not None
        and SELLER_MIN <= price <= BUYER_MAX
    )

print("Negotiation state initialized.")
print("Maximum rounds:", MAX_ROUNDS)
print("Buyer maximum: $", BUYER_MAX)
print("Seller minimum: $", SELLER_MIN)
print("Initial status:", negotiation_state["st

In [92]:

# Step 103: Negotiation controller regression tests

print("=== CONTROLLER REGRESSION TESTS ===")

# Test 1: Valid Buyer offer accepted by Seller
state = create_negotiation_state()

apply_negotiation_decision(
    state, "buyer",
    {"decision": "OFFER", "price": 410.0,
     "message": "I offer $410."}
)

apply_negotiation_decision(
    state, "seller",
    {"decision": "ACCEPT", "price": 410.0,
     "message": "Accepted."}
)

assert state["status"] == "DEAL"
assert state["agreed_price"] == 410.0
assert len(state["history"]) == 2
print("PASS: Valid agreement at $410")


# Test 2: Buyer cannot exceed $420
state = create_negotiation_state()

try:
    apply_negotiation_decision(
        state, "buyer",
        {"decision": "OFFER", "price": 500.0,
         "message": "I offer $500."}
    )
    raise AssertionError("Invalid Buyer offer was accepted")
except ValueError:
    assert state["history"] == []
    print("PASS: Buyer maximum enforced")


# Test 3: Seller cannot offer below $400
state = create_negotiation_state()

try:
    apply_negotiation_decision(
        state, "seller",
        {"decision": "OFFER", "price": 350.0,
         "message": "I offer $350."}
    )
    raise AssertionError("Invalid Seller offer was accepted")
except ValueError:
    assert state["history"] == []
    print("PASS: Seller minimum enforced")


# Test 4: Acceptance must match outstanding offer
state = create_negotiation_state()

apply_negotiation_decision(
    state, "buyer",
    {"decision": "OFFER", "price": 405.0,
     "message": "I offer $405."}
)

try:
    apply_negotiation_decision(
        state, "seller",
        {"decision": "ACCEPT", "price": 410.0,
         "message": "I accept $410."}
    )
    raise AssertionError("Mismatched acceptance was accepted")
except ValueError:
    assert state["status"] == "ONGOING"
    assert len(state["history"]) == 1
    print("PASS: Exact-offer acceptance enforced")


# Test 5: Walk-away ends negotiation
state = create_negotiation_state()

apply_negotiation_decision(
    state, "buyer",
    {"decision": "WALK_AWAY", "price": None,
     "message": "No agreement."}
)

assert state["status"] == "WALK_AWAY"
print("PASS: Walk-away ends negotiation")

print("\nALL 5 REGRESSION TESTS PASSED.")
print("No API calls made. Saved evidence unchanged.")


=== CONTROLLER REGRESSION TESTS ===
PASS: Valid agreement at $410
PASS: Buyer maximum enforced
PASS: Seller minimum enforced
PASS: Exact-offer acceptance enforced
PASS: Walk-away ends negotiation

ALL 5 REGRESSION TESTS PASSED.
No API calls made. Saved evidence unchanged.


In [93]:

# Step 104: Create a safe submission notebook copy

import os
import shutil

source_path = (
    "/content/drive/MyDrive/Colab Notebooks/"
    "AI_Agents_Assignment.ipynb"
)

submission_dir = (
    "/content/drive/MyDrive/"
    "PS5_Agent_Assignment/submission"
)

destination_path = os.path.join(
    submission_dir,
    "PS5_Multi_Agent_Negotiation_Submission.ipynb"
)

assert os.path.isfile(source_path), (
    "Original notebook not found."
)

os.makedirs(submission_dir, exist_ok=True)

# Never overwrite an existing submission copy.
assert not os.path.exists(destination_path), (
    "Submission copy already exists. "
    "Do not overwrite it."
)

shutil.copy2(source_path, destination_path)

assert os.path.isfile(destination_path)

print("=== SUBMISSION COPY CREATED ===")
print("Original:", source_path)
print("Submission:", destination_path)
print("Original size:", os.path.getsize(source_path), "bytes")
print("Copy size:", os.path.getsize(destination_path), "bytes")
print("Original preserved:", os.path.isfile(source_path))
print("\nNo Anthropic API calls made.")


=== SUBMISSION COPY CREATED ===
Original: /content/drive/MyDrive/Colab Notebooks/AI_Agents_Assignment.ipynb
Submission: /content/drive/MyDrive/PS5_Agent_Assignment/submission/PS5_Multi_Agent_Negotiation_Submission.ipynb
Original size: 340876 bytes
Copy size: 340876 bytes
Original preserved: True

No Anthropic API calls made.


In [94]:

# Step 105: Identify potential live API execution cells

import ast

print("=== POTENTIAL LIVE API EXECUTION CELLS ===")

for index, cell in enumerate(cells):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))

    try:
        tree = ast.parse(source)
    except SyntaxError:
        continue

    # Detect API calls in executable top-level code,
    # excluding function definitions.
    top_level_code = ast.Module(
        body=[
            node for node in tree.body
            if not isinstance(
                node, (
                    ast.FunctionDef,
                    ast.AsyncFunctionDef,
                    ast.ClassDef
                )
            )
        ],
        type_ignores=[]
    )

    found = []

    for node in ast.walk(top_level_code):
        if not isinstance(node, ast.Call):
            continue

        try:
            call_name = ast.unparse(node.func)
        except Exception:
            continue

        if (
            call_name == "query"
            or call_name.endswith(".messages.create")
            or call_name in (
                "run_full_negotiation",
                "run_five_agent_workflow",
                "run_live_research"
            )
        ):
            found.append(call_name)

    if found:
        first_line = (
            source.splitlines()[0]
            if source.splitlines() else ""
        )
        print(
            f"Cell {index:03d}: {first_line[:65]}"
        )
        print("  Calls:", sorted(set(found)))

print("\nInspection complete.")
print("No API calls made. No files modified.")


=== POTENTIAL LIVE API EXECUTION CELLS ===
Cell 002: 
  Calls: ['client.messages.create']
Cell 043: 
  Calls: ['query']
Cell 044: 
  Calls: ['query']
Cell 051: 
  Calls: ['query']
Cell 056: 
  Calls: ['query']
Cell 070: 
  Calls: ['run_full_negotiation']
Cell 073: 
  Calls: ['run_full_negotiation']
Cell 075: 
  Calls: ['query']

Inspection complete.
No API calls made. No files modified.


In [95]:

# Step 106: Find top-level async execution cells

import ast

print("=== TOP-LEVEL ASYNC EXECUTION AUDIT ===")

for index, cell in enumerate(cells):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))

    try:
        tree = ast.parse(source)
    except SyntaxError:
        continue

    # Inspect executable statements, excluding function bodies
    top_level_tree = ast.Module(
        body=[
            node for node in tree.body
            if not isinstance(
                node, (
                    ast.FunctionDef,
                    ast.AsyncFunctionDef,
                    ast.ClassDef
                )
            )
        ],
        type_ignores=[]
    )

    found = [
        type(node).__name__
        for node in ast.walk(top_level_tree)
        if isinstance(node, (ast.Await, ast.AsyncFor))
    ]

    if found:
        first_line = next(
            (
                line.strip()
                for line in source.splitlines()
                if line.strip()
            ),
            "(empty)"
        )

        print(f"Cell {index:03d}: {first_line[:70]}")
        print("  Async operations:", sorted(set(found)))

print("\nAudit complete.")
print("No API calls made. No files modified.")


=== TOP-LEVEL ASYNC EXECUTION AUDIT ===
Cell 016: # Step 24: Test Claude Agent SDK
  Async operations: ['Await']
Cell 018: # Step 26: Test Coordinator -> Researcher delegation
  Async operations: ['Await']
Cell 022: # Step 30: Test Researcher tool permissions
  Async operations: ['Await']
Cell 026: # Step 34: Test Researcher web access
  Async operations: ['Await']
Cell 028: # Step 37: Test improved Researcher
  Async operations: ['Await']
Cell 031: # Step 40: Test Buyer and Seller SDK delegation
  Async operations: ['Await']
Cell 040: # Step 49: Test one real Buyer response
  Async operations: ['Await']
Cell 041: # Step 50: Test Seller response to real Buyer offer
  Async operations: ['Await']
Cell 043: # Step 52: Configure and test the Referee agent
  Async operations: ['AsyncFor']
Cell 044: # Step 53: Reliable Referee report capture
  Async operations: ['AsyncFor']
Cell 045: # Step 54: Buyer responds in round 2
  Async operations: ['Await']
Cell 046: # Step 55: Seller responds in ro

In [96]:

# Step 107: Classify notebook cells for cleanup

import ast

print("=== NOTEBOOK CLEANUP CLASSIFICATION ===")

categories = {
    "markdown": [],
    "definitions": [],
    "live_or_async": [],
    "other": [],
}

for index, cell in enumerate(cells):
    if cell["cell_type"] == "markdown":
        categories["markdown"].append(index)
        continue

    source = "".join(cell.get("source", []))

    try:
        tree = ast.parse(source)
    except SyntaxError:
        categories["other"].append(index)
        continue

    executable = ast.Module(
        body=[
            node for node in tree.body
            if not isinstance(
                node, (
                    ast.FunctionDef,
                    ast.AsyncFunctionDef,
                    ast.ClassDef
                )
            )
        ],
        type_ignores=[]
    )

    has_async = any(
        isinstance(node, (ast.Await, ast.AsyncFor))
        for node in ast.walk(executable)
    )

    has_definitions = any(
        isinstance(
            node, (ast.FunctionDef, ast.AsyncFunctionDef)
        )
        for node in tree.body
    )

    if has_async:
        categories["live_or_async"].append(index)
    elif has_definitions:
        categories["definitions"].append(index)
    else:
        categories["other"].append(index)

for category, indices in categories.items():
    print(f"\n{category.upper()} ({len(indices)} cells)")
    print(indices)

print("\nClassification only; no cells modified.")
print("No API calls made.")


=== NOTEBOOK CLEANUP CLASSIFICATION ===

MARKDOWN (3 cells)
[5, 8, 11]

DEFINITIONS (23 cells)
[3, 7, 19, 21, 25, 33, 34, 35, 36, 37, 38, 39, 48, 60, 65, 66, 67, 68, 69, 79, 82, 83, 85]

LIVE_OR_ASYNC (24 cells)
[16, 18, 22, 26, 28, 31, 40, 41, 43, 44, 45, 46, 49, 50, 51, 52, 53, 54, 55, 56, 63, 70, 73, 75]

OTHER (40 cells)
[0, 1, 2, 4, 6, 9, 10, 12, 13, 14, 15, 17, 20, 23, 24, 27, 29, 30, 32, 42, 47, 57, 58, 59, 61, 62, 64, 71, 72, 74, 76, 77, 78, 80, 81, 84, 86, 87, 88, 89]

Classification only; no cells modified.
No API calls made.


In [97]:

# Step 108: Inspect agent and research setup

inspect_cells = [17, 25, 27, 29, 30, 33, 81, 83]

print("=== AGENT CONFIGURATION AUDIT ===")

for index in inspect_cells:
    source = "".join(cells[index].get("source", []))

    print("\n" + "=" * 60)
    print(f"CELL {index}")
    print("=" * 60)
    print(source)

print("\nInspection complete.")
print("No API calls made. No files modified.")


=== AGENT CONFIGURATION AUDIT ===

CELL 17

# Step 25: Define our five-agent architecture

from claude_agent_sdk import (
    AgentDefinition,
    ClaudeAgentOptions
)

agents = {
    "researcher": AgentDefinition(
        description="Researches used PS5 market prices.",
        prompt="""
        You are a market research specialist.
        Research current used PlayStation 5 prices.
        Use web search to find real market evidence.
        Report prices, sources, and a fair price range.
        Never invent prices or sources.
        """,
        tools=["WebSearch"]
    ),

    "buyer": AgentDefinition(
        description="Negotiates to buy a used PS5.",
        prompt=buyer_prompt_v3,
        tools=[]
    ),

    "seller": AgentDefinition(
        description="Negotiates to sell a used PS5.",
        prompt=seller_prompt_v3,
        tools=[]
    ),

    "referee": AgentDefinition(
        description="Checks negotiation rules and outcomes.",
        prompt="""
        You are 

In [98]:

# Step 109: Inspect saved research and evidence fields

import json
import os

evidence_dir = (
    "/content/drive/MyDrive/"
    "PS5_Agent_Assignment/evidence"
)

for filename in (
    "original_405_run.json",
    "automated_401_run.json"
):
    path = os.path.join(evidence_dir, filename)

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    print(f"\n=== {filename} ===")
    print("Top-level fields:", list(data.keys()))

    for field in ("research_report", "market_context"):
        value = data.get(field)
        print(
            f"{field}:",
            f"{len(value)} characters"
            if isinstance(value, str)
            else "not present as a top-level string"
        )

    negotiation = data.get("negotiation", {})
    print("Negotiation status:", negotiation.get("status"))
    print("Agreed price:", negotiation.get("agreed_price"))

print("\nRead-only inspection complete.")
print("No API calls made. No files modified.")



=== original_405_run.json ===
Top-level fields: ['run_type', 'negotiation', 'sdk_events', 'referee_report']
research_report: not present as a top-level string
market_context: not present as a top-level string
Negotiation status: DEAL
Agreed price: 405.0

=== automated_401_run.json ===
Top-level fields: ['run_type', 'negotiation', 'sdk_events', 'referee_report', 'limits']
research_report: not present as a top-level string
market_context: not present as a top-level string
Negotiation status: DEAL
Agreed price: 401.0

Read-only inspection complete.
No API calls made. No files modified.


In [99]:

# Step 110: Save existing Researcher evidence

import json
import os

evidence_dir = (
    "/content/drive/MyDrive/"
    "PS5_Agent_Assignment/evidence"
)

research_path = os.path.join(
    evidence_dir,
    "ps5_market_research.json"
)

assert isinstance(research_report, str)
assert research_report.strip()
assert isinstance(market_context, str)
assert market_context.strip()

# Preserve existing evidence rather than overwriting it.
assert not os.path.exists(research_path), (
    "Research evidence file already exists. "
    "Inspect it before overwriting."
)

research_evidence = {
    "source": "Anthropic Agent SDK Researcher with WebSearch",
    "research_report": research_report,
    "market_context": market_context,
    "verification_note": (
        "Low-confidence market research leads. "
        "Individual sold listings were not independently "
        "verified; do not present estimates as confirmed "
        "completed-sale averages."
    )
}

os.makedirs(evidence_dir, exist_ok=True)

with open(research_path, "w", encoding="utf-8") as f:
    json.dump(research_evidence, f, indent=2)

# Verify the saved file can be read.
with open(research_path, "r", encoding="utf-8") as f:
    saved = json.load(f)

assert saved["research_report"] == research_report
assert saved["market_context"] == market_context

print("=== RESEARCH EVIDENCE SAVED ===")
print("File:", research_path)
print("Report characters:", len(research_report))
print("Context characters:", len(market_context))
print("File size:", os.path.getsize(research_path), "bytes")
print("PASS: Research preserved successfully.")
print("No Anthropic API calls made.")


=== RESEARCH EVIDENCE SAVED ===
File: /content/drive/MyDrive/PS5_Agent_Assignment/evidence/ps5_market_research.json
Report characters: 1826
Context characters: 2116
File size: 4284 bytes
PASS: Research preserved successfully.
No Anthropic API calls made.


In [100]:

# Step 111: Load saved market research independently

import json
import os

research_path = (
    "/content/drive/MyDrive/"
    "PS5_Agent_Assignment/evidence/"
    "ps5_market_research.json"
)

with open(research_path, "r", encoding="utf-8") as f:
    saved_research = json.load(f)

loaded_research_report = saved_research["research_report"]
loaded_market_context = saved_research["market_context"]

assert loaded_research_report.strip()
assert loaded_market_context.strip()

# Compare against the current working runtime
assert loaded_research_report == research_report
assert loaded_market_context == market_context

print("=== SAVED RESEARCH LOAD TEST ===")
print("Report loaded:", len(loaded_research_report), "characters")
print("Context loaded:", len(loaded_market_context), "characters")
print("Report matches original:", True)
print("Context matches original:", True)
print("Verification note:", saved_research["verification_note"])

print("\nPASS: Research can be loaded from saved evidence.")
print("No API calls made. No files modified.")


=== SAVED RESEARCH LOAD TEST ===
Report loaded: 1826 characters
Context loaded: 2116 characters
Report matches original: True
Context matches original: True
Verification note: Low-confidence market research leads. Individual sold listings were not independently verified; do not present estimates as confirmed completed-sale averages.

PASS: Research can be loaded from saved evidence.
No API calls made. No files modified.


In [103]:

# Step 112: Submission notebook safety check

import json
import os

submission_path = (
    "/content/drive/MyDrive/"
    "PS5_Agent_Assignment/submission/"
    "PS5_Multi_Agent_Negotiation_Submission.ipynb"
)

evidence_dir = (
    "/content/drive/MyDrive/"
    "PS5_Agent_Assignment/evidence"
)

assert os.path.isfile(submission_path)

with open(submission_path, "r", encoding="utf-8") as f:
    submission_notebook = json.load(f)

assert len(submission_notebook["cells"]) == 90

required_evidence = [
    "original_405_run.json",
    "automated_401_run.json",
    "ps5_market_research.json",
]

for filename in required_evidence:
    path = os.path.join(evidence_dir, filename)
    assert os.path.isfile(path), f"Missing: {filename}"
    print("PASS:", filename)

print("\n=== SUBMISSION NOTEBOOK ===")
print("Notebook:", os.path.basename(submission_path))
print("Original cell count:", len(submission_notebook["cells"]))
print("Evidence files:", len(required_evidence))
print("Notebook format:", submission_notebook["nbformat"])

print("\nPASS: Ready for controlled restructuring.")
print("No API calls made. No files modified.")


AssertionError: 

In [104]:

# Step 113: Compare original and submission notebook

import json
import os

original_path = (
    "/content/drive/MyDrive/Colab Notebooks/"
    "AI_Agents_Assignment.ipynb"
)

submission_path = (
    "/content/drive/MyDrive/PS5_Agent_Assignment/"
    "submission/PS5_Multi_Agent_Negotiation_Submission.ipynb"
)

for label, path in [
    ("ORIGINAL", original_path),
    ("SUBMISSION", submission_path),
]:
    print(f"\n=== {label} ===")
    print("Exists:", os.path.isfile(path))

    if os.path.isfile(path):
        with open(path, "r", encoding="utf-8") as f:
            notebook = json.load(f)

        code_count = sum(
            c["cell_type"] == "code"
            for c in notebook["cells"]
        )
        markdown_count = sum(
            c["cell_type"] == "markdown"
            for c in notebook["cells"]
        )

        print("Total cells:", len(notebook["cells"]))
        print("Code cells:", code_count)
        print("Markdown cells:", markdown_count)
        print("File size:", os.path.getsize(path), "bytes")

print("\nRead-only comparison complete.")



=== ORIGINAL ===
Exists: True
Total cells: 104
Code cells: 101
Markdown cells: 3
File size: 377567 bytes

=== SUBMISSION ===
Exists: True
Total cells: 95
Code cells: 92
Markdown cells: 3
File size: 340876 bytes

Read-only comparison complete.


In [105]:

# Step 114: Inspect submission cells added after the original 90

import json

with open(submission_path, "r", encoding="utf-8") as f:
    submission_notebook = json.load(f)

submission_cells = submission_notebook["cells"]

print("=== SUBMISSION CELLS 90–94 ===")

for index in range(90, len(submission_cells)):
    cell = submission_cells[index]
    source = "".join(cell.get("source", []))

    print(f"\nCELL {index}")
    print("Type:", cell["cell_type"])
    print("First 500 characters:")
    print(source[:500] if source.strip() else "[EMPTY]")

print("\nRead-only inspection complete.")
print("No API calls made. No files modified.")


=== SUBMISSION CELLS 90–94 ===

CELL 90
Type: code
First 500 characters:

# Step 99: Inspect assignment notebook cells (read-only)

import json
import os

notebook_path = (
    "/content/drive/MyDrive/Colab Notebooks/"
    "AI_Agents_Assignment.ipynb"
)

assert os.path.isfile(notebook_path), "Notebook not found."

with open(notebook_path, "r", encoding="utf-8") as f:
    notebook_data = json.load(f)

cells = notebook_data.get("cells", [])

code_count = sum(c["cell_type"] == "code" for c in cells)
markdown_count = sum(c["cell_type"] == "markdown" for c in cells)

prin

CELL 91
Type: code
First 500 characters:

# Step 100: Find function definitions and duplicates

import ast
from collections import defaultdict

definitions = defaultdict(list)

for index, cell in enumerate(cells):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))

    try:
        tree = ast.parse(source)
    except SyntaxError:
        continue

    for node in tree.bo

In [106]:

# Step 115: Create clean submission notebook skeleton

import json
import os

clean_path = (
    "/content/drive/MyDrive/"
    "PS5_Agent_Assignment/submission/"
    "PS5_Multi_Agent_Negotiation_CLEAN.ipynb"
)

assert not os.path.exists(clean_path), (
    "Clean notebook already exists; refusing to overwrite it."
)

def markdown_cell(text):
    return {
        "cell_type": "markdown",
        "metadata": {},
        "source": text.splitlines(keepends=True)
    }

sections = [
    ("# Multi-Agent Marketplace Negotiator\n"
     "**Track A — Used PlayStation 5 (Disc Edition)**\n\n"
     "Five-agent architecture using the Anthropic Claude Agent SDK.\n\n"
     "The notebook separates saved experiment evidence from "
     "optional paid live demonstrations."),

    ("## 1. Setup and Authentication\n"
     "Install dependencies and configure the Anthropic API key "
     "using Colab Secrets."),

    ("## 2. Agent Architecture and Permissions\n"
     "Coordinator, Researcher, Buyer, Seller, and Referee. "
     "Only Researcher may use WebSearch."),

    ("## 3. Market Research\n"
     "Load preserved Researcher findings, including limitations. "
     "An optional live search can refresh the evidence."),

    ("## 4. Negotiation Controller\n"
     "Adversarial negotiation with offer validation, "
     "convergence detection, and a six-round hard cap."),

    ("## 5. SDK Event Logging and Execution\n"
     "Record agent messages, tool calls, delegation, "
     "and final outcomes."),

    ("## 6. Three Tuning Iterations\n"
     "Document V1, V2, and V3 with actual observed results. "
     "Early tuning used the direct Anthropic API; "
     "final demonstrations used the Agent SDK."),

    ("## 7. Saved Results and Referee Validation\n"
     "Load and verify genuine $405 and $401 negotiation runs."),

    ("## 8. Optional Live Five-Agent Demonstration\n"
     "Run only when explicitly enabled. "
     "This section may incur Anthropic API charges."),

    ("## 9. Conclusion and Limitations\n"
     "Summarize architecture, outcomes, market-data confidence, "
     "and reproducibility.")
]

clean_notebook = {
    "cells": [markdown_cell(s) for s in sections],
    "metadata": {
        "kernelspec": {
            "display_name": "Python 3",
            "language": "python",
            "name": "python3"
        },
        "language_info": {"name": "python"},
        "colab": {"provenance": []}
    },
    "nbformat": 4,
    "nbformat_minor": 5
}

os.makedirs(os.path.dirname(clean_path), exist_ok=True)

with open(clean_path, "w", encoding="utf-8") as f:
    json.dump(clean_notebook, f, indent=2)

print("=== CLEAN NOTEBOOK CREATED ===")
print("File:", clean_path)
print("Section headings:", len(clean_notebook["cells"]))
print("Existing notebooks untouched.")
print("No API calls made.")


=== CLEAN NOTEBOOK CREATED ===
File: /content/drive/MyDrive/PS5_Agent_Assignment/submission/PS5_Multi_Agent_Negotiation_CLEAN.ipynb
Section headings: 10
Existing notebooks untouched.
No API calls made.


In [107]:

import json
from pathlib import Path

original = Path(
    "/content/drive/MyDrive/Colab Notebooks/AI_Agents_Assignment.ipynb"
)
clean = Path(
    "/content/drive/MyDrive/PS5_Agent_Assignment/submission/"
    "PS5_Multi_Agent_Negotiation_CLEAN.ipynb"
)

assert original.exists(), "Original notebook missing"
assert clean.exists(), "Clean notebook missing"

source = json.loads(original.read_text(encoding="utf-8"))
target = json.loads(clean.read_text(encoding="utf-8"))

print("Original notebook:", len(source["cells"]), "cells")
print("Clean notebook:", len(target["cells"]), "cells")
print("Both notebooks are accessible.")
print("Ready to transfer working code.")


Original notebook: 107 cells
Clean notebook: 10 cells
Both notebooks are accessible.
Ready to transfer working code.


In [108]:

# Build a populated submission notebook from genuine saved work.
# No Anthropic API calls. Original and CLEAN notebooks untouched.

import json
from pathlib import Path

root = Path("/content/drive/MyDrive/PS5_Agent_Assignment")
original_path = Path(
    "/content/drive/MyDrive/Colab Notebooks/AI_Agents_Assignment.ipynb"
)
clean_path = root / "submission/PS5_Multi_Agent_Negotiation_CLEAN.ipynb"
output_path = root / "submission/PS5_Multi_Agent_Negotiation_POPULATED.ipynb"
evidence_dir = root / "evidence"

assert not output_path.exists(), "Populated notebook already exists."
assert original_path.exists() and clean_path.exists()

required = [
    "original_405_run.json",
    "automated_401_run.json",
    "ps5_market_research.json",
]
assert all((evidence_dir / name).exists() for name in required)

original = json.loads(original_path.read_text(encoding="utf-8"))
clean = json.loads(clean_path.read_text(encoding="utf-8"))

def md(text):
    return {
        "cell_type": "markdown",
        "metadata": {},
        "source": text.splitlines(keepends=True),
    }

def code(text):
    return {
        "cell_type": "code",
        "execution_count": None,
        "metadata": {},
        "outputs": [],
        "source": text.splitlines(keepends=True),
    }

cells = []

cells.append(md(
    "# Multi-Agent Marketplace Negotiator\n\n"
    "**Track A: Used PS5 Disc Edition with Controller**\n\n"
    "Five roles: Coordinator, Researcher, Buyer, Seller, Referee.\n\n"
    "This notebook preserves genuine development experiments and "
    "provides executable checks for saved SDK results. "
    "Archived experiment code is not automatically re-executed."
))

cells.append(md(
    "## 1. Reproducible Evidence Review\n\n"
    "The following cells do not require an Anthropic API key "
    "and do not make paid requests. Google Drive must be mounted."
))

cells.append(code('''
from pathlib import Path
import json

from google.colab import drive
drive.mount("/content/drive")

EVIDENCE_DIR = Path(
    "/content/drive/MyDrive/PS5_Agent_Assignment/evidence"
)

def load_evidence(filename):
    path = EVIDENCE_DIR / filename
    assert path.is_file(), f"Missing evidence: {path}"
    return json.loads(path.read_text(encoding="utf-8"))

research = load_evidence("ps5_market_research.json")
run_405 = load_evidence("original_405_run.json")
run_401 = load_evidence("automated_401_run.json")

print("Saved research and both SDK runs loaded.")
print("Research characters:", len(research["research_report"]))
'''.strip()))

cells.append(code('''
for label, record, expected_price in [
    ("Original SDK run", run_405, 405),
    ("Automated SDK run", run_401, 401),
]:
    negotiation = record["negotiation"]
    assert negotiation["status"] == "DEAL"
    assert float(negotiation["agreed_price"]) == expected_price
    assert 400 <= float(negotiation["agreed_price"]) <= 420
    assert record.get("sdk_events"), f"Missing SDK events: {label}"
    assert record.get("referee_report"), f"Missing referee report: {label}"

    print(label)
    print("  Status:", negotiation["status"])
    print("  Agreed price: $", negotiation["agreed_price"])
    print("  Logged SDK events:", len(record["sdk_events"]))
    print("  Referee report present: YES")

print("\\nPASS: Saved negotiation evidence validated.")
'''.strip()))

cells.append(md(
    "## 2. Research Quality and Limitations\n\n"
    "The Researcher used WebSearch through the Claude Agent SDK. "
    "Its findings are low-confidence market leads; individual "
    "completed sales were not independently verified. "
    "The research must not be represented as a confirmed "
    "market average."
))

cells.append(code('''
print(research["research_report"])
print("\\nLIMITATION:")
print(research["verification_note"])
'''.strip()))

cells.append(md(
    "## 3. Three Tuning Iterations\n\n"
    "The original development notebook records V1, V2, and V3. "
    "**These early tuning experiments used the direct Anthropic API**, "
    "not the Claude Agent SDK. Their original logs follow."
))

for idx in (5, 8, 11):
    source_cell = original["cells"][idx]
    assert source_cell["cell_type"] == "markdown"
    cells.append(source_cell)

cells.append(md(
    "## 4. Original Implementation and SDK Experiment Archive\n\n"
    "The following is the genuine development code, preserved "
    "in original order. Code is displayed rather than executed "
    "automatically to prevent duplicate definitions, accidental "
    "paid API calls, and dependence on prior runtime state.\n\n"
    "**Important:** This archive is supporting evidence, not "
    "a verified clean-run implementation."
))

archive_count = 0
for idx, source_cell in enumerate(original["cells"]):
    if idx in (5, 8, 11):
        continue

    source_text = "".join(source_cell.get("source", []))
    if not source_text.strip():
        continue

    if source_cell["cell_type"] == "code":
        # Indented code blocks avoid conflicts with Markdown fences.
        archived = "\n".join(
            "    " + line for line in source_text.splitlines()
        )
        cells.append(md(
            f"### Original development cell {idx}\n\n"
            f"{archived}\n"
        ))
        archive_count += 1

cells.append(md(
    "## 5. Conclusion\n\n"
    "The recorded Claude Agent SDK runs reached valid agreements "
    "at $405 and $401. The implementation includes Researcher-only "
    "WebSearch permission controls, multi-round negotiation, "
    "offer validation, event logging, and Referee review.\n\n"
    "**Remaining reproducibility task:** Extract and verify a "
    "single clean, live SDK workflow from the archived implementation. "
    "The saved-evidence checks above are executable independently "
    "of paid API calls."
))

populated = {
    "cells": cells,
    "metadata": clean["metadata"],
    "nbformat": 4,
    "nbformat_minor": 5,
}

output_path.write_text(
    json.dumps(populated, indent=2),
    encoding="utf-8",
)

# Verify the file was written and is readable.
verified = json.loads(output_path.read_text(encoding="utf-8"))
assert len(verified["cells"]) == len(cells)

print("=== POPULATED NOTEBOOK CREATED ===")
print("File:", output_path)
print("Total cells:", len(cells))
print("Archived original code cells:", archive_count)
print("Executable evidence cells:", 3)
print("Original and CLEAN notebooks untouched.")
print("No Anthropic API calls made.")


=== POPULATED NOTEBOOK CREATED ===
File: /content/drive/MyDrive/PS5_Agent_Assignment/submission/PS5_Multi_Agent_Negotiation_POPULATED.ipynb
Total cells: 118
Archived original code cells: 106
Executable evidence cells: 3
Original and CLEAN notebooks untouched.
No Anthropic API calls made.


In [ ]:
# Create an executable working copy for final cleanup
import json
from pathlib import Path

source_path = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "AI_Agents_Assignment.ipynb"
)

final_path = Path(
    "/content/drive/MyDrive/PS5_Agent_Assignment/"
    "submission/PS5_Multi_Agent_Negotiation_FINAL.ipynb"
)

assert source_path.exists(), "Original notebook not found"
assert not final_path.exists(), "FINAL file already exists; refusing to overwrite"

notebook = json.loads(source_path.read_text(encoding="utf-8"))

# Preserve original executable code and genuine outputs.
# Do not execute any cells or make API calls.
final_path.write_text(
    json.dumps(notebook, indent=2),
    encoding="utf-8"
)

saved = json.loads(final_path.read_text(encoding="utf-8"))

print("FINAL working copy created:", final_path)
print("Total cells:", len(saved["cells"]))
print("Executable code cells:", sum(
    c["cell_type"] == "code" for c in saved["cells"]
))
print("Original notebook untouched.")
print("No API calls made.")